# CS781 Phase 4 — judgment prediction from your Phase-3 models

**Kaggle T4 ×2 · seven adaptation experiments · complete-document predictions**

Attach your saved **phase-3-outputs** dataset. This notebook restores your four own foundation families, including learned Phase-3 low-rank updates; predicts integer **0/1** judgments; and produces seven individual submission ZIPs. The three attached assignment notebooks informed the recurrent attention, Transformer, PEFT and distillation mechanisms. Their graded blanks are not filled here.

The score-oriented default learns the new task head, adapts each encoder on final-window evidence, then recomputes complete-document features and fits a hierarchical classifier. A whole-document joint refinement of the strongest candidate checks for an additional development improvement. Full-document joint training for all seven is available through `two_stage=False` at substantially greater compute. All original case text is retained in the final document representation.

The current brief lists **October 21, 2026, 11:59 PM IST** as the deadline. Training and local ZIP generation do not upload to the course platform. Actual scores and total runtime depend on your data and trained weights; no target score is guaranteed.

**Course-use note:** the [brief](https://docs.google.com/document/d/1olqoT5KPhf5CSJLBNjXGyKyyyQDBr6jcnWqPUh5cNk4/edit) restricts LLM-written complete model architectures and tuning and requires an LLM declaration. This notebook contains AI-written code; preserve attribution and follow those course requirements.

In [ ]:
import os,sys,importlib.util,subprocess
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['OMP_NUM_THREADS']='1'
requirements={'tokenizers':'tokenizers>=0.20,<0.24','huggingface_hub':'huggingface_hub>=0.27,<3',
              'sklearn':'scikit-learn>=1.4,<2','matplotlib':'matplotlib>=3.8,<4'}
missing=[spec for name,spec in requirements.items() if importlib.util.find_spec(name) is None]
if missing:subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
import torch
if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])<(2,3):
    raise RuntimeError('Use a current Kaggle PyTorch image (2.3+); keep its installed CUDA/PyTorch pairing.')
print('PyTorch:',torch.__version__)

## Configuration

Select **GPU T4 ×2** in Kaggle. The usual input edit is `PHASE3_ROOT`; `DATA_ROOT=None` downloads only official data using the `HF_TOKEN` secret. There are no external model downloads.

`two_stage=True` is the research-supported compute default. The final hierarchy uses complete documents. Encoder adaptation uses final-window evidence explicitly; it is not equivalent to seven fully joint document runs. Full joint refinement runs for the best candidate by default and is accepted only after a dev improvement. Epoch limits and early stopping remain visible; there is no elapsed-time cutoff that silently discards training.

Keep `PROJECT` and `SETTINGS` fixed to resume an interrupted run. A new design needs a new project directory. Refit, section auditing and an extra ensemble are separate switches after the seven base submissions exist.

In [ ]:
from pathlib import Path
import json,importlib,zipfile,shutil,csv
from IPython.display import display,FileLink
WORKING=Path('/kaggle/working')
PROJECT=WORKING/'capstone_phase4'
CODE=PROJECT/'code'
CODE.mkdir(parents=True,exist_ok=True)
PHASE3_ROOT=Path('/kaggle/input/datasets/hardik4tiwari/phase-3-outputs/capstone_phase3')
DATA_ROOT=None  # Or Path('/kaggle/input/YOUR-OFFICIAL-DATA/Phase_4_CJPE').
DATA_REVISION=None  # Resolve once and pin the official dataset commit.
ALLOW_SELECTION_FALLBACK=True
PARENT_OVERRIDES={}  # Optional architecture -> full-sized compatible Phase-3 best.pt inside PHASE3_ROOT.
SETTINGS={
    'gpus':2,'two_stage':True,
    'chunk_tokens':512,'overlap':64,'chunk_batch':16,
    'head_dim':192,'head_batch':16,
    'batch_docs':4,'accumulation':4,
    'head_epochs':16,'head_patience':3,
    'adapt_epochs':3,'adapt_patience':2,
    'student_layers':4,'allow_warm_fallback':True,
    'joint_epochs':2,'joint_patience':1,'joint_accumulation':4,
}
RUN_BENCHMARK=True
RUN_GPU_SMOKE=True
USE_LEXICAL_FUSION=True
JOINT_REFINE_TOP_K=1
REFIT_TOP_K=0  # Optional all-label refit adds compute; original held-out-model ZIPs are retained.
MAKE_BONUS_ENSEMBLE=True  # An extra ZIP only if dev improves; not an extra required adaptation method.
RUN_SECTION_AUDIT=False  # Three additional dev passes; enable for the report if time permits.
EVALUATE_SUPPLIED_TEST_LABELS=True  # Reporting only, after all development-based choices are final.
MAKE_FULL_WORKING_ARCHIVE=False  # Opt in after checking free disk space.
if torch.cuda.device_count()!=2:raise RuntimeError('Choose Kaggle GPU T4 x2 before Run All.')
for index in range(2):
    gpu=torch.cuda.get_device_properties(index)
    print(index,gpu.name,round(gpu.total_memory/1024**3,2),'GiB; capability',gpu.major,gpu.minor)
torch.set_num_threads(1)
sys.path.insert(0,str(CODE))

## Direct input tree

```text
capstone_phase3/
    parents/tokenizer.json
    parents/decoder.pt
    parents/encoder_decoder.pt
    parents/local_gqa.pt
    parents/hybrid.pt
    runs/01_decoder_full/refit/best.pt
    runs/02_encoder_decoder_full/refit/best.pt
    runs/03_local_gqa_lora/refit/best.pt
    runs/04_hybrid_rslora/refit/best.pt
```

Known `selection/best.pt` files can substitute for missing refits. Every source is printed and checked. A dataset containing only Phase-3 prediction ZIPs is insufficient: learned checkpoints and dense parents are required. Your original Phase-1/2 datasets do not need separate attachment when this complete Phase-3 output structure is available.

```text
Phase_4_CJPE/
    Train/train.json
    Train/dev.json
    Test/test.json
```

Keep official IDs exactly as supplied. Output `prediction.json` is a list of `{"id":"case_id","pred":0}` objects, one per test case.

## Embedded runtime

These cells write all runtime source, including the original four-backbone definitions and the unchanged official evaluator. PyTorch's native recurrent kernels and attention are used where appropriate. Trained weights remain your own throughout.

In [ ]:
MODULE_SOURCE=r'''
"""Four randomly initialized legal language models; no downloaded model weights."""
import math
from dataclasses import dataclass, asdict
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


@dataclass
class ModelConfig:
    architecture: str = "decoder"
    vocab_size: int = 32768
    dim: int = 512
    heads: int = 8
    kv_heads: int = 8
    layers: int = 8
    ff_dim: int = 2048
    encoder_layers: int = 0
    rnn_layers: int = 0
    local_window: int = 0
    max_positions: int = 4096
    dropout: float = 0.0
    gradient_checkpointing: bool = False


def architecture_config(name, **overrides):
    c = ModelConfig(architecture=name)
    if name == "decoder":
        pass
    elif name == "local_gqa":
        c.kv_heads, c.layers, c.local_window = 2, 9, 256
    elif name == "hybrid":
        c.layers, c.rnn_layers = 7, 2
    elif name == "encoder_decoder":
        c.layers, c.encoder_layers = 4, 3
    else:
        raise ValueError(f"Unknown architecture: {name}")
    for k, v in overrides.items():
        if not hasattr(c, k):
            raise ValueError(k)
        setattr(c, k, v)
    assert c.dim % c.heads == 0 and c.heads % c.kv_heads == 0
    assert (c.dim // c.heads) % 2 == 0
    return c


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Reduction in FP32 is useful with T4 FP16 training.
        y = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.weight.to(x.dtype)


class Rotary(nn.Module):
    def __init__(self, head_dim, max_positions):
        super().__init__()
        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        angles = torch.outer(torch.arange(max_positions).float(), inv)
        self.register_buffer("cos", angles.cos(), persistent=False)
        self.register_buffer("sin", angles.sin(), persistent=False)

    def forward(self, x, offset=0):
        n = x.size(-2)
        if offset + n > self.cos.size(0):
            raise ValueError("Position limit exceeded; increase max_positions in the model config.")
        cos = self.cos[offset:offset + n].to(x.dtype)[None, None]
        sin = self.sin[offset:offset + n].to(x.dtype)[None, None]
        a, b = x[..., 0::2], x[..., 1::2]
        return torch.stack((a * cos - b * sin, a * sin + b * cos), dim=-1).flatten(-2)


class Attention(nn.Module):
    def __init__(self, c, causal=True, cross=False):
        super().__init__()
        self.heads, self.kv_heads = c.heads, c.kv_heads
        self.hd = c.dim // c.heads
        self.q = nn.Linear(c.dim, c.dim, bias=False)
        self.k = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.v = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.out = nn.Linear(c.dim, c.dim, bias=False)
        self.rotary = None if cross else Rotary(self.hd, c.max_positions)
        self.causal, self.cross = causal, cross
        self.window = c.local_window if causal and not cross else 0
        self.dropout = c.dropout

    def forward(self, x, memory=None, past=None, position=0, use_cache=False):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.heads, self.hd).transpose(1, 2)
        if self.cross and past is not None:
            k, v = past  # Encoder K/V projections are also cached during decoding.
        else:
            source = x if memory is None else memory
            s = source.size(1)
            k = self.k(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            v = self.v(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            if self.rotary is not None:
                k = self.rotary(k, position)
            if past is not None:
                k, v = torch.cat((past[0], k), -2), torch.cat((past[1], v), -2)
        if self.rotary is not None:
            q = self.rotary(q, position)
        new_cache = (k, v) if use_cache else None
        # Portable GQA: project/cache fewer KV heads, repeat only for the SDPA call.
        # This is not a block-sparse implementation or a guaranteed training speedup.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k = k.repeat_interleave(repeats, dim=1)
            v = v.repeat_interleave(repeats, dim=1)
        mask, is_causal = None, False
        if self.causal:
            s = k.size(-2)
            if self.window or (past is not None and t > 1):
                qi = torch.arange(s - t, s, device=x.device)[:, None]
                ki = torch.arange(s, device=x.device)[None, :]
                mask = ki <= qi
                if self.window:
                    mask = mask & ((qi - ki) < self.window)
            elif past is None:
                is_causal = True
            # Cached single-token query must see all prior keys. is_causal=True
            # here would incorrectly use the upper-left non-square mask.
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=is_causal,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, -1))
        return y, new_cache


class SwiGLU(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gate = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.up = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.down = nn.Linear(c.ff_dim, c.dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class Block(nn.Module):
    def __init__(self, c, encoder=False):
        super().__init__()
        self.attn_norm, self.ff_norm = RMSNorm(c.dim), RMSNorm(c.dim)
        self.attn = Attention(c, causal=not encoder)
        self.ff = SwiGLU(c)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x, past=None, position=0, use_cache=False):
        y, cache = self.attn(self.attn_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache) if use_cache else x


class CrossBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.self_norm, self.cross_norm, self.ff_norm = (RMSNorm(c.dim) for _ in range(3))
        self.self_attn = Attention(c)
        self.cross_attn = Attention(c, causal=False, cross=True)
        self.ff, self.drop = SwiGLU(c), nn.Dropout(c.dropout)

    def forward(self, x, memory, past=None, cross_past=None, position=0, use_cache=False):
        y, cache = self.self_attn(self.self_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        y, cross_cache = self.cross_attn(self.cross_norm(x), memory=memory, past=cross_past, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache, cross_cache) if use_cache else x


class LegalLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.embedding = nn.Embedding(c.vocab_size, c.dim)
        # Output projection uses embedding.weight; there is no second vocabulary matrix.
        if c.architecture == "encoder_decoder":
            self.encoder = nn.ModuleList(Block(c, encoder=True) for _ in range(c.encoder_layers))
            self.encoder_norm = RMSNorm(c.dim)
            self.blocks = nn.ModuleList(CrossBlock(c) for _ in range(c.layers))
        else:
            self.blocks = nn.ModuleList(Block(c) for _ in range(c.layers))
        if c.rnn_layers:
            self.rnn = nn.LSTM(c.dim, c.dim, num_layers=c.rnn_layers, batch_first=True)
            self.rnn_norm = RMSNorm(c.dim)
        self.final_norm = RMSNorm(c.dim)
        self.apply(self._init)
        # Scale residual projections to keep initial residual variance controlled.
        std = 0.02 / math.sqrt(2 * c.layers)
        for name, p in self.named_parameters():
            if name.endswith(("attn.out.weight", "self_attn.out.weight", "cross_attn.out.weight", "ff.down.weight")):
                nn.init.normal_(p, mean=0.0, std=std)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.LSTM):
            for name, p in m.named_parameters():
                if "weight" in name:
                    nn.init.normal_(p, mean=0.0, std=0.02)
                else:
                    nn.init.zeros_(p)

    def encode(self, source_ids):
        x = self.embedding(source_ids)
        for block in self.encoder:
            if self.config.gradient_checkpointing and self.training:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return self.encoder_norm(x)

    def forward(self, input_ids, source_ids=None, state=None, use_cache=False):
        state = state or {}
        position = state.get("position", 0)
        x = self.embedding(input_ids)
        new_state = {"position": position + input_ids.size(1), "self": []}
        if self.config.rnn_layers:
            y, recurrent_state = self.rnn(x, state.get("rnn"))
            x = x + self.rnn_norm(y)
            new_state["rnn"] = recurrent_state
        if self.config.architecture == "encoder_decoder":
            memory = state.get("memory")
            if memory is None:
                if source_ids is None:
                    raise ValueError("encoder_decoder requires source_ids on the first call.")
                memory = self.encode(source_ids)
            new_state.update(memory=memory, cross=[])
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    cp = state.get("cross", [None] * len(self.blocks))[i]
                    x, cache, cc = block(x, memory, p, cp, position, True)
                    new_state["self"].append(cache)
                    new_state["cross"].append(cc)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, memory, use_reentrant=False)
                else:
                    x = block(x, memory)
        else:
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    x, cache = block(x, p, position, True)
                    new_state["self"].append(cache)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, use_reentrant=False)
                else:
                    x = block(x)
        x = self.final_norm(x)
        return (x, new_state) if use_cache else x

    def project(self, hidden):
        return F.linear(hidden, self.embedding.weight)


def parameter_count(model):
    # PyTorch de-duplicates shared Parameter objects here.
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def parameter_formula(c):
    d, h, kv = c.dim, c.dim // c.heads, c.kv_heads
    attention = 2 * d * d + 2 * d * kv * h
    ff = 3 * d * c.ff_dim
    total = c.vocab_size * d + d
    if c.architecture == "encoder_decoder":
        total += c.encoder_layers * (attention + ff + 2 * d) + d
        total += c.layers * (2 * attention + ff + 3 * d)
    else:
        total += c.layers * (attention + ff + 2 * d)
    if c.rnn_layers:
        total += c.rnn_layers * (8 * d * d + 8 * d) + d
    return total


def make_batch(blocks, architecture, prefix_fraction=0.80, objective="full"):
    # blocks has T+1 tokens. The target suffix is identical across architectures.
    t = blocks.size(1) - 1
    split = max(1, min(t, int(t * prefix_fraction)))
    if architecture == "encoder_decoder":
        source = blocks[:, :split]
        inputs, targets = blocks[:, split - 1:-1], blocks[:, split:]
    else:
        source = None
        inputs, targets = blocks[:, :-1], blocks[:, 1:].clone()
        if objective == "continuation":
            targets[:, :split - 1] = -100
    return inputs, targets, source


def language_model_loss(hidden, targets, weight, chunk_tokens=256, checkpoint_chunks=False):
    """Sum/token count lets validation aggregate the true token-weighted NLL."""
    hidden, targets = hidden.reshape(-1, hidden.size(-1)), targets.reshape(-1)
    total = hidden.new_zeros((), dtype=torch.float32)
    count = (targets != -100).sum()
    def loss_chunk(h, y, w):
        # Keep logits in autocast precision until the bounded FP32 CE conversion.
        return F.cross_entropy(F.linear(h, w).float(), y, ignore_index=-100, reduction="sum")
    for i in range(0, targets.numel(), chunk_tokens):
        h, y = hidden[i:i + chunk_tokens], targets[i:i + chunk_tokens]
        if checkpoint_chunks and torch.is_grad_enabled():
            total = total + checkpoint(loss_chunk, h, y, weight, use_reentrant=False)
        else:
            total = total + loss_chunk(h, y, weight)
    return total / count.clamp_min(1), total.detach(), count.detach()


if __name__ == "__main__":
    for name in ("decoder", "encoder_decoder", "local_gqa", "hybrid"):
        c = architecture_config(name)
        with torch.device("meta"):
            model = LegalLM(c)
        n = parameter_count(model)
        assert n == parameter_formula(c) and 50_000_000 <= n <= 300_000_000
        print(name, f"{n:,}", asdict(c))
'''
(CODE/'legal_models.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','legal_models.py')

In [ ]:
MODULE_SOURCE=r'''
"""Phase-4 input validation, complete token coverage, disk-backed bags and exact metrics."""
import hashlib,json,math,os,shutil,zipfile
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset
from tokenizers import Tokenizer
from sklearn.metrics import f1_score,precision_score,recall_score,confusion_matrix

DATA_FILES=['Train/train.json','Train/dev.json','Test/test.json']
def digest(value):return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as handle:
        for chunk in iter(lambda:handle.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
def read_json(path):
    def unique(pairs):
        result={}
        for key,value in pairs:
            if key in result:raise ValueError(f'Duplicate JSON key: {key}')
            result[key]=value
        return result
    return json.loads(Path(path).read_text(encoding='utf-8-sig'),object_pairs_hook=unique)
def atomic_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8');tmp.replace(path)
def atomic_torch(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');torch.save(value,tmp);tmp.replace(path)
def alias(row,names):
    keys=[name for name in names if name in row]
    if len(keys)>1 and any(row[k]!=row[keys[0]] for k in keys[1:]):raise ValueError(f'Conflicting fields: {keys}')
    return row[keys[0]] if keys else None
def shape(value):
    if isinstance(value,dict):return {str(k):type(v).__name__ for k,v in list(value.items())[:8]}
    return {'type':type(value).__name__,'length':len(value) if isinstance(value,list) else None}
def resolve_training_rows(rows,audit=None,reserved_ids=None,split="train"):
    # Repeated source-case IDs can describe distinct training instances. IDs below are internal only.
    raw_counts={};groups={};unique=[];seen=set();case_labels={};text_labels={}
    for row in rows:
        source=row['id'];raw_counts[source]=raw_counts.get(source,0)+1
        text_sha=hashlib.sha256(row['text'].encode('utf-8')).hexdigest();key=(source,text_sha,row['gold'])
        case_labels.setdefault(source,set()).add(row['gold']);text_labels.setdefault((source,text_sha),set()).add(row['gold'])
        if key in seen:continue
        seen.add(key);groups[source]=groups.get(source,0)+1;unique.append((row,text_sha))
    external=set(reserved_ids or ());reserved=set(raw_counts)|external;result=[]
    for row,text_sha in unique:
        source=row['id'];did=source
        if groups[source]>1 or source in external:
            stem=source+'::'+split+'::'+digest([text_sha,row['gold']])[:16];did=stem;counter=0
            while did in reserved:counter+=1;did=stem+'::'+str(counter)
            reserved.add(did)
        result.append({**row,'id':did,'source_id':source})
    if audit is not None:
        disagreements=[source for source,labels in case_labels.items() if len(labels)>1]
        audit.update(raw_rows=len(rows),unique_source_case_ids=len(raw_counts),retained_distinct_rows=len(result),
            repeated_source_id_groups=sum(n>1 for n in raw_counts.values()),exact_copies_removed=len(rows)-len(result),
            distinct_instance_case_groups=sum(n>1 for n in groups.values()),
            internally_named_training_rows=sum(row['id']!=row['source_id'] for row in result),
            source_cases_with_both_labels=len(disagreements),both_label_case_id_examples=disagreements[:20],
            same_text_conflicting_label_groups=sum(len(labels)>1 for labels in text_labels.values()),
            policy=f'Collapse exact (source ID, text, label) copies; preserve differing text/labels as distinct internal {split} rows. Never majority-vote labels.')
    return result
def records(value,split,audit=None,reserved_ids=None):
    if isinstance(value,dict) and set(value) in ({'data'},{'records'},{'documents'}):value=next(iter(value.values()))
    if isinstance(value,list):rows=[(None,r) for r in value]
    elif isinstance(value,dict):
        text=alias(value,('text','sentences'));ids=alias(value,('id','ids','case_id','doc_id'))
        labels=alias(value,('label','labels','gold','outcome'))
        if isinstance(ids,str) and text is not None:rows=[(None,{'id':ids,'text':text,'label':labels})]
        elif isinstance(ids,list) and isinstance(text,list):
            if len(ids)!=len(text) or (labels is not None and (not isinstance(labels,list) or len(labels)!=len(ids))):
                raise ValueError('ID, text and label columns have different lengths.')
            rows=[(None,{'id':did,'text':text[i],'label':None if labels is None else labels[i]}) for i,did in enumerate(ids)]
        elif isinstance(text,dict):
            for column in (ids,labels):
                if column is not None and (not isinstance(column,dict) or set(column)!=set(text)):raise ValueError('JSON column keys differ.')
            rows=[(None,{'id':key if ids is None else ids[key],'text':body,'label':None if labels is None else labels[key]}) for key,body in text.items()]
        else:rows=list(value.items())
    else:raise ValueError(f'{split}: unsupported JSON structure: {shape(value)}')
    result=[];seen=set()
    for key,row in rows:
        if isinstance(row,str) and key is not None:row={'id':key,'text':row}
        if not isinstance(row,dict):raise ValueError(f'{split}: document must be an object; structure={shape(value)}')
        did=alias(row,('id','case_id','doc_id'));did=key if did is None else did
        if not isinstance(did,str) or not did.strip():raise ValueError(f'{split}: original nonempty string IDs are required; IDs are never invented.')
        if key is not None and key!=did:raise ValueError('Mapping key and explicit case ID differ.')
        seen.add(did);text=alias(row,('text','sentences'))
        if isinstance(text,list) and all(isinstance(s,str) for s in text):text='\n'.join(text)
        if not isinstance(text,str):raise ValueError(f'{did}: text must be a string or an ordered list of strings.')
        gold=None if split=='test' else alias(row,('label','labels','gold','outcome'))
        if split=='test':
            gold=None  # Supplied test labels are read only by the separate post-training evaluator.
        elif type(gold) is not int or gold not in (0,1):raise ValueError(f'{did}: expected a scalar integer label 0 or 1.')
        result.append({'id':did,'text':text,'gold':gold,'split':split})
    if not result:raise ValueError(f'{split}: empty split.')
    if split in ('train','dev'):return resolve_training_rows(result,audit,reserved_ids,split=split)
    # The evaluator accepts one prediction per original test case, not one per source row.
    grouped={};copies=0
    for row in result:
        entry=grouped.setdefault(row['id'],{'row':row,'texts':[],'seen':set(),'rows':0})
        entry['rows']+=1
        if row['text'] in entry['seen']:copies+=1;continue
        entry['seen'].add(row['text']);entry['texts'].append(row['text'])
    merged=[{**v['row'],'source_id':did,'text':'\n\n'.join(v['texts'])} for did,v in grouped.items()]
    if audit is not None:
        audit.update(raw_rows=len(result),unique_source_case_ids=len(merged),exact_text_copies_removed=copies,
            repeated_source_id_groups=sum(v['rows']>1 for v in grouped.values()),
            cases_with_distinct_texts_combined=sum(len(v['texts'])>1 for v in grouped.values()),
            policy='One original test ID per prediction; collapse exact text copies and concatenate distinct case texts in source order. Test labels are excluded from model data; supplied labels are evaluated separately after training.')
    return merged
def load_records(root,return_audit=False):
    root=Path(root);test_audit={};dev_audit={};audit={}
    test=records(read_json(root/'Test/test.json'),'test',test_audit)
    test_ids={row['source_id'] for row in test}
    dev=records(read_json(root/'Train/dev.json'),'dev',dev_audit,reserved_ids=test_ids)
    dev_sources={row['source_id'] for row in dev};held_out=dev_sources|test_ids
    excluded_dev=[row for row in dev if row['source_id'] in test_ids]
    dev=[row for row in dev if row['source_id'] not in test_ids]
    if not dev:raise ValueError('No development records remain after test source-case protection.')
    dev_audit.update(test_overlap_rows_removed=len(excluded_dev),
        test_overlap_source_cases_removed=len({row['source_id'] for row in excluded_dev}),
        test_overlap_case_id_examples=sorted({row['source_id'] for row in excluded_dev})[:20],
        scoring_unit='Distinct (source case ID, text, label) development records; no label voting.',
        retained_development_rows=len(dev))
    train=records(read_json(root/'Train/train.json'),'train',audit,
        reserved_ids=held_out|{row['id'] for row in dev}|test_ids)
    removed=[row for row in train if row['source_id'] in held_out]
    audit['held_out_case_rows_removed']=len(removed)
    audit['held_out_source_case_ids_removed']=len({row['source_id'] for row in removed})
    audit['held_out_case_id_examples']=sorted({row['source_id'] for row in removed})[:20]
    train=[row for row in train if row['source_id'] not in held_out]
    if not train:raise ValueError('No training cases remain after held-out source-case filtering.')
    audit['training_rows_after_source_case_filter']=len(train)
    audit['development_id_resolution']=dev_audit;audit['test_id_resolution']=test_audit
    result=train+dev+test
    if len({row['id'] for row in result})!=len(result):raise ValueError('Internal record IDs unexpectedly collide.')
    return (result,audit) if return_audit else result
def prepare_data(root,tokenizer_path,output,chunk_tokens=512,overlap=64):
    root=Path(root);output=Path(output);output.mkdir(parents=True,exist_ok=True)
    if not 0<=overlap<chunk_tokens:raise ValueError('Require 0 <= overlap < chunk_tokens.')
    identity=digest({'files':{name:file_hash(root/name) for name in DATA_FILES},'tokenizer':file_hash(tokenizer_path),
        'chunk_tokens':chunk_tokens,'overlap':overlap,'version':2})
    manifest=output/'manifest.json'
    if manifest.exists():
        old=read_json(manifest)
        if old['fingerprint']!=identity:raise ValueError('Data/tokenization changed; use a new PROJECT.')
        if any(old['docs'][i]['gold'] is not None for i in old['splits']['test']):raise ValueError('Unsafe cache contains test targets. Use a new PROJECT; never train or refit with test labels.')
        if (output/'tokens.i32').stat().st_size!=4*old['total_tokens']:raise ValueError('Incomplete token cache.')
        return str(manifest)
    rows,record_audit=load_records(root,return_audit=True);dev_hash={digest(' '.join(r['text'].split())) for r in rows if r['split']=='dev'}
    removed=[r['id'] for r in rows if r['split']=='train' and digest(' '.join(r['text'].split())) in dev_hash]
    tokenizer=Tokenizer.from_file(str(tokenizer_path));tokenizer.no_padding();tokenizer.no_truncation()
    unk=tokenizer.token_to_id('<unk>');unk=0 if unk is None else unk
    docs=[];token_count=chunk_count=empty=0;temporary=output/'tokens.i32.tmp'
    with temporary.open('wb') as handle:
        for index,row in enumerate(rows):
            ids=tokenizer.encode(row['text'],add_special_tokens=False).ids;empty+=int(not ids);ids=ids or [unk]
            np.asarray(ids,dtype='<i4').tofile(handle);n=len(ids);windows=[]
            final=max(0,n-chunk_tokens);starts=list(range(0,final+1,chunk_tokens-overlap))
            if starts[-1]!=final:starts.append(final)
            for start in starts:
                length=min(chunk_tokens,n-start);windows.append([token_count+start,length,start/max(1,n-1)])
            docs.append({'id':row['id'],'gold':row['gold'],'split':row['split'],'token_count':n,'chunks':windows,
                'feature_start':chunk_count,'text_sha256':digest(row['text']),'source_id':row.get('source_id',row['id'])})
            token_count+=n;chunk_count+=len(windows)
            if (index+1)%2000==0:print('Tokenized cases:',index+1,flush=True)
    temporary.replace(output/'tokens.i32')
    splits={s:[i for i,d in enumerate(docs) if d['split']==s and d['id'] not in removed] for s in ('train','dev','test')}
    splits['all']=splits['train']+splits['dev'];splits['every']=list(range(len(docs)))
    if any(not splits[s] for s in ('train','dev','test')):raise ValueError('Deduplication left an empty split.')
    if set(docs[i]['gold'] for i in splits['train'])!={0,1}:raise ValueError('Both binary classes are required in training.')
    value={'fingerprint':identity,'docs':docs,'splits':splits,'total_tokens':token_count,'total_chunks':chunk_count,
        'chunk_tokens':chunk_tokens,'overlap':overlap,'tokenizer_sha256':file_hash(tokenizer_path),
        'audit':{'empty_documents':empty,'training_dev_duplicates_removed':removed,'training_id_resolution':record_audit,'all_tokens_retained':True,
            'case_counts':{s:len(v) for s,v in splits.items()},'max_chunks_per_case':max(len(d['chunks']) for d in docs),
            'label_support':{s:{str(y):sum(docs[i]['gold']==y for i in splits[s]) for y in (0,1)} for s in ('train','dev')}}}
    atomic_json(manifest,value);return str(manifest)

class Bags(Dataset):
    def __init__(self,manifest,split,features=None,teacher=None,view='all'):
        self.path=Path(manifest);self.data=read_json(self.path);self.rows=self.data['splits'][split]
        self.tokens=np.memmap(self.path.parent/'tokens.i32',dtype='<i4',mode='r')
        self.features=None
        if features:
            info=read_json(Path(features)/'manifest.json')
            if not info['complete'] or info['data_fingerprint']!=self.data['fingerprint']:raise ValueError('Frozen feature cache identity/completion mismatch.')
            self.features=np.memmap(Path(features)/'features.f16',dtype='<f2',mode='r',shape=tuple(info['shape']))
        if view not in ('all','last'):raise ValueError('Unknown training context view.')
        self.teacher,self.view=teacher,view
    def __len__(self):return len(self.rows)
    def __getitem__(self,index):return self.document(self.rows[index])
    def document(self,index):
        doc=self.data['docs'][index];windows=doc['chunks'];start=doc['feature_start']
        if self.view=='last':start+=len(windows)-1;windows=windows[-1:]
        return {'index':index,'id':doc['id'],'chunks':[np.array(self.tokens[a:a+n],dtype=np.int64) for a,n,p in windows] if self.features is None else None,
            'features':torch.from_numpy(np.array(self.features[start:start+len(windows)],dtype=np.float32)) if self.features is not None else None,
            'position':[[p,1-p,p*p,float(j==0),float(j==len(windows)-1),math.log1p(n)/8] for j,(a,n,p) in enumerate(windows)],
            'target':doc['gold'],'teacher':self.teacher[doc['id']] if self.teacher is not None else None}
def collate(rows):
    B=len(rows);S=max(len(r['position']) for r in rows);positions=torch.zeros(B,S,6);mask=torch.zeros(B,S,dtype=torch.bool)
    owners=[];chunks=[];features=None
    if rows[0]['features'] is not None:features=torch.zeros(B,S,rows[0]['features'].shape[-1])
    for b,row in enumerate(rows):
        n=len(row['position']);positions[b,:n]=torch.tensor(row['position']);mask[b,:n]=True
        if features is not None:features[b,:n]=row['features']
        else:
            chunks.extend(row['chunks']);owners.extend(b*S+j for j in range(n))
    return {'chunks':chunks,'owners':owners,'features':features,'position':positions,'mask':mask,
        'target':torch.tensor([r['target'] if r['target'] is not None else -100 for r in rows]),
        'teacher':torch.stack([torch.as_tensor(r['teacher']) for r in rows]) if rows[0]['teacher'] is not None else None,
        'rows':rows}
def to_device(batch,device):return {k:v.to(device,non_blocking=True) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}
def metric(y,probability,threshold=.5):
    pred=(np.asarray(probability)>=threshold).astype(int);y=np.asarray(y)
    return {'macro_f1':float(f1_score(y,pred,average='macro',labels=[0,1],zero_division=0)),
        'precision':float(precision_score(y,pred,average='binary',pos_label=1,zero_division=0)),
        'recall':float(recall_score(y,pred,average='binary',pos_label=1,zero_division=0)),
        'confusion_matrix':confusion_matrix(y,pred,labels=[0,1]).tolist(),'threshold':float(threshold)}
def choose_calibration(y,neural,lexical=None):
    best=None
    for weight in ((0.,.15,.3) if lexical is not None else (0.,)):
        p=np.asarray(neural)*(1-weight)+(np.asarray(lexical)*weight if lexical is not None else 0)
        for threshold in (.5,.45,.55,.4,.6):
            scores=metric(y,p,threshold)
            if best is None or scores['macro_f1']>best[0]['macro_f1']+1e-10:best=(scores,{'lexical_weight':weight,'threshold':threshold})
    return best
def probabilities(values):
    x=np.asarray(values,dtype=np.float64);x-=x.max(-1,keepdims=True);e=np.exp(x);return e[:,1]/e.sum(-1)
def write_submission(ids,preds,path):
    if len(ids)!=len(set(ids)) or len(ids)!=len(preds):raise ValueError('Submission ID/count mismatch.')
    if any(type(y) is not int or y not in (0,1) for y in preds):raise ValueError('Predictions must be integer 0/1.')
    records=[{'id':did,'pred':y} for did,y in zip(ids,preds)];raw=json.dumps(records,ensure_ascii=False).encode()
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix('.tmp')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as archive:archive.writestr('prediction.json',raw)
    tmp.replace(path);return records,hashlib.sha256(raw).hexdigest()
'''
(CODE/'cj_data.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','cj_data.py')

In [ ]:
MODULE_SOURCE=r'''
"""Own Phase-3 checkpoint restoration and full-document CJPE classifiers."""
import gc,math,shutil,types
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from legal_models import LegalLM,ModelConfig,Attention
from cj_data import atomic_json,atomic_torch,read_json,file_hash,digest

SOURCES={'decoder':'01_decoder_full','encoder_decoder':'02_encoder_decoder_full','local_gqa':'03_local_gqa_lora','hybrid':'04_hybrid_rslora'}
def fold_phase3(saved,parent):
    cfg=saved['config'];kind=cfg['method']
    if kind not in ('full','lora','rslora','lorma','ia3'):raise ValueError('Choose a full-sized, foldable Phase-3 source, not its KD student.')
    state={k[len('backbone.'):]:v.float() for k,v in saved['model'].items() if k.startswith('backbone.')}
    if not state:raise ValueError('Phase-3 checkpoint has no learned backbone weights.')
    dense={k:v.float().clone() for k,v in parent['model'].items()};covered=set()
    for key,value in state.items():
        if key in dense:dense[key]=value;covered.add(key)
        elif key.endswith('.base.weight'):
            dest=key[:-len('.base.weight')]+'.weight'
            if dest not in dense:raise ValueError(f'Unexpected wrapped parent weight: {key}')
            dense[dest]=value;covered.add(dest)
        elif not key.endswith(('.A.weight','.B.weight','.scale')):raise ValueError(f'Unrecognized Phase-3 backbone field: {key}')
    if kind in ('lora','rslora','lorma'):
        rank=cfg.get('rank',8);scale=cfg.get('alpha',16.)/(math.sqrt(rank) if kind=='rslora' else rank)
        prefixes={key[:-len('.A.weight')] for key in state if key.endswith('.A.weight')}
        if not prefixes:raise ValueError('Phase-3 low-rank checkpoint contains no adaptation factors.')
        for name in prefixes:
            A=state[name+'.A.weight'];B=state[name+'.B.weight'];W=dense[name+'.weight']
            dense[name+'.weight']=W+scale*(B@(A@W) if kind=='lorma' else B@A)
    elif kind=='ia3':
        for key,scale in state.items():
            if not key.endswith('.scale'):continue
            name=key[:-len('.scale')];W=dense[name+'.weight']
            dense[name+'.weight']=W*(scale[None,:] if name.rsplit('.',1)[-1]=='down' else scale[:,None])
    if not saved.get('delta_only',False) and covered!=set(dense):raise ValueError('Dense Phase-3 checkpoint omits foundation weights.')
    return dense
def restore_phase3(root,output,allow_selection=True,overrides=None,allow_tiny=False):
    root=Path(root);output=Path(output);output.mkdir(parents=True,exist_ok=True);overrides=overrides or {}
    if not root.is_dir():raise FileNotFoundError(f'Attach phase-3-outputs and set PHASE3_ROOT to its capstone_phase3 folder: {root}')
    if set(overrides)-set(SOURCES):raise ValueError('Unknown architecture in PARENT_OVERRIDES.')
    source_tokenizer=root/'parents/tokenizer.json'
    if not source_tokenizer.is_file():raise FileNotFoundError(f'Keep the complete Phase-3 outputs, including {source_tokenizer}')
    tokenizer_hash=file_hash(source_tokenizer);tokenizer=output/'tokenizer.json'
    if tokenizer.exists() and file_hash(tokenizer)!=tokenizer_hash:raise ValueError('Tokenizer changed; use a new PROJECT.')
    if not tokenizer.exists():shutil.copy2(source_tokenizer,tokenizer)
    parents={};audit={}
    for architecture,name in SOURCES.items():
        refit=root/'runs'/name/'refit/best.pt';selection=root/'runs'/name/'selection/best.pt'
        source=Path(overrides[architecture]) if architecture in overrides else refit if refit.is_file() else selection if allow_selection else refit
        base=root/'parents'/f'{architecture}.pt'
        if not source.resolve().is_relative_to(root.resolve()):raise ValueError('Parent overrides must stay inside PHASE3_ROOT.')
        if not source.is_file() or not base.is_file():raise FileNotFoundError(f'Need both {source} and {base}. Keep parents/ with compact checkpoints.')
        sha=file_hash(source);base_sha=file_hash(base);target=output/f'{architecture}.pt'
        if target.exists():
            previous=torch.load(target,map_location='cpu',weights_only=False)
            if previous['provenance']['phase3_sha256']!=sha or previous['provenance']['phase3_base_sha256']!=base_sha:
                raise ValueError('A Phase-3 parent changed; use a new PROJECT.')
            del previous
        else:
            saved=torch.load(source,map_location='cpu',weights_only=False);parent=torch.load(base,map_location='cpu',weights_only=False)
            if saved.get('training_origin')!='own_phase1_phase2_phase3_rr' or saved['config']['architecture']!=architecture:
                raise ValueError(f'{source}: expected your own matching Phase-3 RR model.')
            provenance=saved['provenance'];origin=parent['provenance']
            for key in ('phase1_sha256','phase2_sha256','tokenizer_sha256'):
                if provenance.get(key)!=origin.get(key):raise ValueError(f'Phase-3/parent lineage mismatch: {key}')
            if origin['tokenizer_sha256']!=tokenizer_hash:raise ValueError('Tokenizer does not match the saved lineage.')
            config=parent['config'];state=fold_phase3(saved,parent);model=LegalLM(ModelConfig(**config));model.load_state_dict(state,strict=True)
            count=sum(p.numel() for p in model.parameters())
            if not allow_tiny and not 50_000_000<=count<=300_000_000:raise ValueError('Foundation parent must have 50M–300M parameters.')
            atomic_torch(target,{'model':state,'config':config,'provenance':{**origin,'phase3_sha256':sha,
                'phase3_base_sha256':base_sha,'phase3_checkpoint':str(source),'phase3_method':saved['config']['method'],
                'source_stage':saved['stage'],'source_parameters':count}})
            del saved,parent,state,model;gc.collect()
        parents[architecture]=str(target)
        audit[architecture]={'source':str(source),'sha256':sha,'base_sha256':base_sha,'selection_fallback':source==selection}
    atomic_json(output/'input_manifest.json',{'parents':audit,'tokenizer_sha256':tokenizer_hash,'input_root':str(root)})
    return parents,str(tokenizer)

class LoRMA(nn.Module):
    def __init__(self,base,rank,alpha):
        super().__init__();self.base=base;self.scale=alpha/rank;base.requires_grad_(False)
        self.A=nn.Linear(base.out_features,rank,bias=False);self.B=nn.Linear(rank,base.out_features,bias=False)
        nn.init.normal_(self.A.weight,std=.02);nn.init.zeros_(self.B.weight)
    def forward(self,x):
        y=self.base(x);return y+self.scale*self.B(self.A(y))
class Bottleneck(nn.Module):
    def __init__(self,width,bottleneck):
        super().__init__();self.norm=nn.LayerNorm(width);self.down=nn.Linear(width,bottleneck);self.up=nn.Linear(bottleneck,width)
        nn.init.zeros_(self.up.weight);nn.init.zeros_(self.up.bias)
    def forward(self,x):return self.up(F.gelu(self.down(self.norm(x))))
def adapter_hook(module,args,value):
    if not isinstance(value,torch.Tensor):raise ValueError('Adapters require classification mode without generation caches.')
    return value+module.task_adapter(value)
def masked_attention(self,x,memory=None,past=None,position=0,use_cache=False):
    if past is not None or use_cache:raise ValueError('CJPE uses complete classification chunks without generation caches.')
    b,t,_=x.shape;source=x if memory is None else memory;s=source.shape[1]
    q=self.q(x).view(b,t,self.heads,self.hd).transpose(1,2)
    k=self.k(source).view(b,s,self.kv_heads,self.hd).transpose(1,2)
    v=self.v(source).view(b,s,self.kv_heads,self.hd).transpose(1,2)
    if self.rotary is not None:q=self.rotary(q);k=self.rotary(k)
    if self.kv_heads!=self.heads:k=k.repeat_interleave(self.heads//self.kv_heads,1);v=v.repeat_interleave(self.heads//self.kv_heads,1)
    valid=self._cj_valid;causal=self.causal;mask=None
    if self.window or not self._cj_all_valid:
        key=(t,s,self.window,causal,x.device)
        if getattr(self,'_cj_allowed_key',None)!=key:
            qi=torch.arange(t,device=x.device)[:,None];ki=torch.arange(s,device=x.device)[None,:]
            allowed=ki<=qi if causal else torch.ones(t,s,device=x.device,dtype=torch.bool)
            if self.window:allowed&=(qi-ki)<self.window
            self._cj_allowed_key=key;self._cj_allowed=allowed
        allowed=self._cj_allowed
        mask=allowed[None,None]&valid[:,None,None,:s];causal=False
    out=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,is_causal=causal,dropout_p=self.dropout if self.training else 0.)
    return self.out(out.transpose(1,2).contiguous().view(b,t,-1)),None

class JudgmentModel(nn.Module):
    def __init__(self,backbone,cfg):
        super().__init__();self.backbone=backbone;self.cfg=cfg;d=backbone.config.dim;h=cfg.get('head_dim',192)
        self.pool_dim=2*d;method=cfg['method'];backbone.requires_grad_(False)
        if method in ('full','kd','gradual'):backbone.requires_grad_(True)
        elif method=='partial':
            for block in list(backbone.blocks)[-cfg.get('unfrozen_layers',2):]:block.requires_grad_(True)
            backbone.final_norm.requires_grad_(True)
        elif method=='lorma':
            targets={'q','k','v','out','gate','up','down'}
            replacements=[(name,m) for name,m in backbone.named_modules() if isinstance(m,nn.Linear) and name.rsplit('.',1)[-1] in targets]
            for name,m in replacements:
                parent,attr=name.rsplit('.',1);setattr(backbone.get_submodule(parent),attr,LoRMA(m,cfg.get('rank',8),cfg.get('alpha',16.)))
        elif method=='adapter':
            blocks=list(backbone.blocks)+list(backbone.encoder) if hasattr(backbone,'encoder') else list(backbone.blocks)
            for block in blocks:
                block.task_adapter=Bottleneck(d,cfg.get('adapter_dim',48));block.register_forward_hook(adapter_hook)
        elif method=='bitfit':
            biases=[p for name,p in backbone.named_parameters() if 'bias' in name.rsplit('.',1)[-1]]
            if not biases:raise ValueError('This parent has no biases. BitFit uses the hybrid backbone with existing LSTM biases.')
            for p in biases:p.requires_grad_(True)
        else:raise ValueError(f'Unknown method {method}')
        self.attention_modules=[m for m in backbone.modules() if isinstance(m,Attention)]
        for m in self.attention_modules:m.forward=types.MethodType(masked_attention,m);m.dropout=cfg.get('encoder_dropout',.05)
        backbone.config.gradient_checkpointing=False
        for m in backbone.modules():
            if isinstance(m,nn.Dropout):m.p=cfg.get('encoder_dropout',.05)
        if cfg.get('head_only'):backbone.requires_grad_(False)
        self.project=nn.Sequential(nn.LayerNorm(2*d+6),nn.Linear(2*d+6,h),nn.GELU(),nn.Dropout(cfg.get('dropout',.2)))
        self.head_type=cfg.get('head_type','bigru')
        if self.head_type=='transformer':
            layer=nn.TransformerEncoderLayer(h,4,3*h,dropout=cfg.get('dropout',.2),batch_first=True,norm_first=True,activation='gelu')
            self.context=nn.TransformerEncoder(layer,2,enable_nested_tensor=False)
        else:self.context=nn.GRU(h,h//2,num_layers=2,bidirectional=True,batch_first=True,dropout=cfg.get('dropout',.2))
        self.norm=nn.LayerNorm(h);self.attention=nn.Sequential(nn.Linear(h,h//2),nn.Tanh(),nn.Linear(h//2,1,bias=False))
        self.classifier=nn.Sequential(nn.LayerNorm(4*h),nn.Dropout(cfg.get('dropout',.2)),nn.Linear(4*h,2))
    def set_epoch(self,epoch):
        if self.cfg['method']!='gradual' or self.cfg.get('head_only'):return
        count=min(len(self.backbone.blocks),self.cfg.get('unfrozen_layers',2)*2**epoch)
        self.backbone.requires_grad_(False)
        for block in list(self.backbone.blocks)[-count:]:block.requires_grad_(True)
        self.backbone.final_norm.requires_grad_(True)
        if count==len(self.backbone.blocks):self.backbone.requires_grad_(True)
    def _hybrid_block(self,x,valid,block):
        # Re-establish this chunk's mask during backward recomputation as well as forward.
        all_valid=bool(valid.all())
        for module in self.attention_modules:module._cj_valid=valid;module._cj_all_valid=all_valid
        return block(x)
    def _hybrid_tokens(self,ids,valid):
        # Keep the fused cuDNN LSTM out of activation-checkpoint recomputation.
        # This is the same residual LSTM -> Transformer computation as LegalLM.forward.
        x=self.backbone.embedding(ids)
        y,_=self.backbone.rnn(x)
        x=x+self.backbone.rnn_norm(y)
        for block in self.backbone.blocks:
            x=checkpoint(self._hybrid_block,x,valid,block,use_reentrant=False)
        return self.backbone.final_norm(x)
    def pool_tokens(self,ids,valid):
        all_valid=bool(valid.all())
        for module in self.attention_modules:module._cj_valid=valid;module._cj_all_valid=all_valid
        hybrid_checkpoint=(self.backbone.config.architecture=='hybrid' and self.training
            and self.cfg.get('encoder_checkpointing',True)
            and any(p.requires_grad for p in self.backbone.parameters()))
        if hybrid_checkpoint:hidden=self._hybrid_tokens(ids,valid)
        else:hidden=self.backbone(ids,source_ids=ids) if self.backbone.config.architecture=='encoder_decoder' else self.backbone(ids)
        mean=(hidden.float()*valid[:,:,None]).sum(1)/valid.sum(1)[:,None]
        last=hidden[torch.arange(len(ids),device=ids.device),valid.sum(1)-1].float()
        return torch.cat((mean,last),-1)
    def encode_chunks(self,chunks,device):
        values=[];step=self.cfg.get('chunk_batch',8)
        for a in range(0,len(chunks),step):
            local=chunks[a:a+step];T=max(map(len,local));ids=torch.zeros(len(local),T,device=device,dtype=torch.long);valid=torch.zeros_like(ids,dtype=torch.bool)
            for i,row in enumerate(local):ids[i,:len(row)]=torch.as_tensor(row.copy(),device=device);valid[i,:len(row)]=True
            checkpoint_encoder=(self.training and self.cfg.get('encoder_checkpointing',True)
                and any(p.requires_grad for p in self.backbone.parameters()))
            if checkpoint_encoder and self.backbone.config.architecture!='hybrid':
                value=checkpoint(self.pool_tokens,ids,valid,use_reentrant=False)
            else:value=self.pool_tokens(ids,valid)
            values.append(value)
        return torch.cat(values)
    def chunk_features(self,batch):
        if batch['features'] is not None:
            if not self.cfg.get('head_only'):raise ValueError('Frozen feature caches are forbidden after backbone adaptation.')
            return batch['features']
        B,S=batch['mask'].shape;vectors=self.encode_chunks(batch['chunks'],batch['mask'].device)
        indices=torch.tensor(batch['owners'],device=vectors.device);pooled=vectors.new_zeros(B*S,self.pool_dim)
        return pooled.index_copy(0,indices,vectors).reshape(B,S,-1)
    def forward(self,batch):
        x=self.project(torch.cat((self.chunk_features(batch),batch['position']),-1));mask=batch['mask'];B,S,_=x.shape
        if self.head_type=='transformer':x=self.context(x,src_key_padding_mask=~mask)
        else:
            packed=nn.utils.rnn.pack_padded_sequence(x,mask.sum(1).cpu(),batch_first=True,enforce_sorted=False)
            x,_=self.context(packed);x,_=nn.utils.rnn.pad_packed_sequence(x,batch_first=True,total_length=S)
        x=self.norm(x);score=self.attention(x).squeeze(-1).float().masked_fill(~mask,-1e9)
        attention=torch.softmax(score,dim=1);global_pool=(x*attention[:,:,None]).sum(1)
        maximum=x.float().masked_fill(~mask[:,:,None],-1e9).max(1).values
        # Complementary section summaries keep beginning and ending evidence available.
        ordinal=torch.arange(S,device=x.device)[None,:];length=mask.sum(1)[:,None]
        first=mask&(ordinal<torch.ceil(length.float()/4));last=mask&(ordinal>=torch.floor(3*length.float()/4))
        section=lambda m:(x*m[:,:,None]).sum(1)/m.sum(1).clamp_min(1)[:,None]
        vector=torch.cat((global_pool,maximum,section(first),section(last)),-1)
        return {'logits':self.classifier(vector),'attention':attention}

def build_model(cfg,checkpoint_path=None):
    parent=torch.load(cfg['parent'],map_location='cpu',weights_only=False);config=ModelConfig(**parent['config']);state=parent['model'];retained=None
    if cfg['method']=='kd':
        n=cfg.get('student_layers',4);original=config.layers
        if config.architecture!='decoder' or not 1<=n<original:raise ValueError('Student must retain fewer decoder blocks than its own parent.')
        retained=[round(i*(original-1)/(n-1)) for i in range(n)] if n>1 else [original-1]
        state={k:v for k,v in state.items() if not k.startswith('blocks.')}
        for new,old in enumerate(retained):
            prefix=f'blocks.{old}.';state.update({f'blocks.{new}.'+k[len(prefix):]:v for k,v in parent['model'].items() if k.startswith(prefix)})
        config.layers=n
    backbone=LegalLM(config);backbone.load_state_dict(state,strict=True);model=JudgmentModel(backbone,cfg)
    if cfg.get('backbone_init'):
        source=torch.load(cfg['backbone_init'],map_location='cpu',weights_only=False)
        if source['config']['method']!=cfg['method'] or source['config']['architecture']!=cfg['architecture']:
            raise ValueError('Encoder-stage initialization has an incompatible adaptation method/family.')
        if source.get('training_origin')!='own_phase1_phase2_phase3_phase4_cjpe':raise ValueError('Expected your own Phase-4 encoder-stage checkpoint.')
        for key in ('phase3_sha256','phase3_base_sha256','tokenizer_sha256'):
            if source['provenance'].get(key)!=parent['provenance'].get(key):raise ValueError('Encoder-stage lineage changed.')
        if source.get('parent_sha256') and source['parent_sha256']!=file_hash(cfg['parent']):raise ValueError('Encoder-stage parent changed.')
        learned={k:v for k,v in source['model'].items() if k.startswith('backbone.')}
        if not learned:raise ValueError('Encoder-stage checkpoint has no adapted backbone weights.')
        missing,unexpected=model.load_state_dict(learned,strict=False)
        if unexpected:raise ValueError('Encoder-stage weights do not match the classifier skeleton.')
    path=checkpoint_path or cfg.get('head_init')
    if path:
        saved=torch.load(path,map_location='cpu',weights_only=False);weights=saved['model']
        if not checkpoint_path:weights={k:v for k,v in weights.items() if not k.startswith('backbone.')}
        missing,unexpected=model.load_state_dict(weights,strict=False)
        required={k for k,p in model.named_parameters() if p.requires_grad} if checkpoint_path else {k for k in model.state_dict() if not k.startswith('backbone.')}
        if unexpected or set(missing)&required:raise ValueError('Incomplete or incompatible CJPE checkpoint.')
    return model,{**parent['provenance'],'retained_parent_blocks':retained,'backbone_config':vars(config)}
def parameter_report(model):return {'total':sum(p.numel() for p in model.parameters()),'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad),
    'backbone':sum(p.numel() for p in model.backbone.parameters())}
def classification_loss(logits,batch,cfg,weights):
    z=logits.float();ce=F.cross_entropy(z,batch['target'],weight=weights,label_smoothing=cfg.get('label_smoothing',.02))
    if cfg['method']!='kd':return ce,{'ce':float(ce.detach())}
    if batch['teacher'] is None:raise ValueError('KD needs real Phase-4 two-class teacher logits, never Phase-3 RR logits.')
    teacher=batch['teacher'].detach().float();tau=cfg.get('temperature',2.);alpha=cfg.get('kd_alpha',.5)
    kl=F.kl_div(F.log_softmax(z/tau,-1),F.softmax(teacher/tau,-1),reduction='batchmean')*tau*tau
    return (1-alpha)*ce+alpha*kl,{'ce':float(ce.detach()),'kd':float(kl.detach())}
'''
(CODE/'cj_model.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','cj_model.py')

In [ ]:
MODULE_SOURCE=r'''
"""Distributed feature extraction, prediction, resumable selection and fixed-epoch refit."""
import argparse,contextlib,gc,json,math,os,random,time
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader,Sampler
from cj_data import Bags,collate,to_device,read_json,atomic_json,atomic_torch,file_hash,digest,metric,probabilities
from cj_model import build_model,parameter_report,classification_loss

def setup():
    rank=int(os.environ.get('RANK',0));world=int(os.environ.get('WORLD_SIZE',1));local=int(os.environ.get('LOCAL_RANK',0))
    device=torch.device('cuda',local) if torch.cuda.is_available() else torch.device('cpu')
    if device.type=='cuda':torch.cuda.set_device(local)
    if world>1 and not dist.is_initialized():
        if device.type=='cuda':dist.init_process_group('nccl',device_id=device)
        else:dist.init_process_group('gloo')
    return rank,world,device
def barrier():
    if dist.is_initialized():dist.barrier()
def amp(device):return torch.autocast('cuda',dtype=torch.float16) if device.type=='cuda' else contextlib.nullcontext()
def signature(cfg):
    identity={'parent':file_hash(cfg['parent']),'student_layers':cfg.get('student_layers') if cfg['method']=='kd' else None,'pool':'mean_last','mask':'inherited'}
    if cfg.get('backbone_init'):identity['adapted_encoder']=file_hash(cfg['backbone_init'])
    return digest(identity)
def rng_state(device):return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
    'cuda':torch.cuda.get_rng_state(device) if device.type=='cuda' else None}
def restore_rng(state,device):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    if device.type=='cuda':torch.cuda.set_rng_state(state['cuda'],device)
class LengthSampler(Sampler):
    def __init__(self,dataset,rank,world,seed):self.dataset,self.rank,self.world,self.seed,self.epoch=dataset,rank,world,seed,0
    def set_epoch(self,epoch):self.epoch=epoch
    def __len__(self):return math.ceil(len(self.dataset)/self.world)
    def __iter__(self):
        generator=torch.Generator().manual_seed(self.seed+self.epoch);order=torch.randperm(len(self.dataset),generator=generator).tolist();bucket=[]
        for first in range(0,len(order),256):
            bucket.extend(sorted(order[first:first+256],key=lambda i:len(self.dataset.data['docs'][self.dataset.rows[i]]['chunks'])))
        extra=len(self)*self.world-len(bucket)
        bucket.extend((bucket*math.ceil(extra/max(1,len(bucket))))[:extra])
        return iter(bucket[self.rank::self.world])

@torch.inference_mode()
def infer_local(model,dataset,rank,world,device,batch_size=1,sections='all'):
    model.eval();values={};attention={};indices=list(range(rank,len(dataset),world))
    # Cached feature batches avoid thousands of tiny GPU launches during head selection.
    for first in range(0,len(indices),batch_size):
        rows=[dataset[i] for i in indices[first:first+batch_size]]
        if sections!='all':
            for row in rows:
                n=len(row['position']);keep=list(range(n))
                if sections=='first':keep=keep[:max(1,math.ceil(n/4))]
                elif sections=='last':keep=keep[-max(1,math.ceil(n/4)):]
                else:raise ValueError(sections)
                row['position']=[row['position'][j] for j in keep]
                if row['features'] is not None:row['features']=row['features'][keep]
                else:row['chunks']=[row['chunks'][j] for j in keep]
        batch=to_device(collate(rows),device)
        with amp(device):out=model(batch)
        for i,row in enumerate(rows):
            values[row['id']]=out['logits'][i].float().cpu()
            if len(attention)<8:attention[row['id']]=out['attention'][i,:len(row['position'])].float().cpu().tolist()
    return values,attention
def gather_values(local,world):
    if world==1:return local
    parts=[None]*world;dist.all_gather_object(parts,local);result={}
    for part in parts:
        if set(part)&set(result):raise ValueError('Inference duplicated a case across ranks.')
        result.update(part)
    return result
def feature_cache(args,cfg,rank,world,device):
    root=Path(args.output);root.mkdir(parents=True,exist_ok=True);dataset=Bags(args.data,'every');data=dataset.data
    model,_=build_model(cfg);model.to(device).eval();width=model.pool_dim
    identity=digest({'data':data['fingerprint'],'encoder':signature(cfg),'shape':[data['total_chunks'],width]})
    manifest=root/'manifest.json';old=read_json(manifest) if manifest.exists() else None
    if old and old['identity']!=identity:raise ValueError('Feature cache source changed.')
    if old and old.get('complete'):
        if (root/'features.f16').stat().st_size!=2*data['total_chunks']*width:raise ValueError('Frozen feature file is incomplete.')
        if rank==0:print('Reusing frozen features:',root,flush=True)
        return
    if rank==0 and old is None:
        needed=2*data['total_chunks']*width
        if shutil_free(root)<needed+256*1024**2:raise RuntimeError(f'Need {needed/1024**3:.2f} GiB free for this complete feature cache. No cases are dropped.')
        with (root/'features.f16').open('wb') as handle:handle.truncate(needed)
        with (root/'done.u8').open('wb') as handle:handle.truncate(len(data['docs']))
        atomic_json(manifest,{'identity':identity,'complete':False,'shape':[data['total_chunks'],width],
            'encoder_signature':signature(cfg),'data_fingerprint':data['fingerprint']})
    barrier();matrix=np.memmap(root/'features.f16',dtype='<f2',mode='r+',shape=(data['total_chunks'],width))
    done=np.memmap(root/'done.u8',dtype='u1',mode='r+',shape=(len(data['docs']),));pending=[];started=time.monotonic();processed=0
    def flush():
        matrix.flush()
        if pending:done[pending]=1;done.flush();pending.clear()
    with torch.inference_mode():
        for index in range(rank,len(data['docs']),world):
            if done[index]:continue
            row=dataset.document(index)
            with amp(device):features=model.encode_chunks(row['chunks'],device)
            doc=data['docs'][index];a=doc['feature_start'];matrix[a:a+len(doc['chunks'])]=features.float().cpu().numpy().astype('<f2')
            pending.append(index);processed+=1
            if len(pending)>=64:flush()
            if processed%500==0 and rank==0:print(json.dumps({'feature_cases_rank0':processed,'elapsed_minutes':(time.monotonic()-started)/60}),flush=True)
    flush();barrier()
    if rank==0:
        if not np.all(done):raise ValueError('Feature extraction omitted cases.')
        info=read_json(manifest);info.update(complete=True,elapsed_seconds=time.monotonic()-started);atomic_json(manifest,info)
    barrier()
def shutil_free(path):
    import shutil
    return shutil.disk_usage(path).free
def predict(args,cfg,rank,world,device):
    saved=torch.load(args.checkpoint,map_location='cpu',weights_only=False);cfg=saved['config'];root=Path(args.output);root.mkdir(parents=True,exist_ok=True)
    dataset=Bags(args.data,args.split,args.features,view=cfg.get('view','all'))
    if dataset.data['fingerprint']!=saved['data_fingerprint']:raise ValueError('Prediction dataset differs from the checkpoint.')
    if args.features:
        if not cfg.get('head_only'):raise ValueError('An adapted checkpoint must recompute its encoder features.')
        if read_json(Path(args.features)/'manifest.json')['encoder_signature']!=signature(cfg):raise ValueError('Wrong frozen feature encoder.')
    identity=digest({'checkpoint':file_hash(args.checkpoint),'data':dataset.data['fingerprint'],'split':args.split,'sections':args.sections})
    complete=root/'complete.pt'
    if complete.exists():
        old=torch.load(complete,map_location='cpu',weights_only=False)
        if old['identity']!=identity:raise ValueError('Prediction cache source changed.')
        if rank==0:print('Reusing logits:',root,flush=True)
        return
    model,_=build_model(cfg,args.checkpoint);model.to(device)
    local,attention=infer_local(model,dataset,rank,world,device,cfg.get('head_batch',16) if args.features else 1,args.sections)
    values=gather_values(local,world)
    if rank==0:
        expected=[dataset.data['docs'][i]['id'] for i in dataset.rows]
        if set(expected)!=set(values):raise ValueError('Prediction cache ID mismatch.')
        atomic_torch(complete,{'identity':identity,'values':values,'split':args.split,'data_fingerprint':dataset.data['fingerprint'],
            'checkpoint_sha256':file_hash(args.checkpoint),'config':cfg,'parameters':saved['parameters'],'attention_examples_rank0':attention})
    barrier()

def train(args,cfg,rank,world,device):
    root=Path(args.output);root.mkdir(parents=True,exist_ok=True);seed=cfg.get('seed',230437)
    random.seed(seed+rank);np.random.seed(seed+rank);torch.manual_seed(seed+rank)
    if bool(args.features)!=bool(cfg.get('head_only')):raise ValueError('Features are required exactly for frozen-backbone head training.')
    teacher=None;data=read_json(args.data)
    if args.features and read_json(Path(args.features)/'manifest.json')['encoder_signature']!=signature(cfg):raise ValueError('Frozen features came from another parent.')
    if args.teacher:
        cached=torch.load(args.teacher,map_location='cpu',weights_only=False)
        if cached.get('smoke_only') and not args.smoke:raise ValueError('Synthetic preflight teachers are forbidden in real training.')
        if cached['data_fingerprint']!=data['fingerprint'] or cached['split']!=args.split:raise ValueError('Teacher cache has the wrong dataset or training split.')
        teacher=cached['values']
    if cfg['method']=='kd' and teacher is None:raise ValueError('KD requires cached Phase-4 binary teacher logits.')
    dataset=Bags(args.data,args.split,args.features,teacher,view=cfg.get('view','all'))
    if teacher is not None and any(dataset.data['docs'][i]['id'] not in teacher for i in dataset.rows):raise ValueError('Teacher cache omits training cases.')
    dev=Bags(args.data,'dev',args.features,view=cfg.get('view','all'));sampler=LengthSampler(dataset,rank,world,seed)
    generator=torch.Generator();loader=DataLoader(dataset,batch_size=cfg.get('batch_docs',1),sampler=sampler,collate_fn=collate,
        num_workers=0,pin_memory=device.type=='cuda',generator=generator)
    model,provenance=build_model(cfg);model.to(device);report=parameter_report(model)
    if cfg['method']=='kd' and args.teacher:
        if cached['teacher_parameters']<=report['total']:raise ValueError('The teacher must have more parameters than this student.')
    # Gradual unfreezing keeps optimizer membership stable; DDP is rebuilt after each policy change.
    parameters=[(n,p) for n,p in model.named_parameters() if p.requires_grad]
    groups=[]
    for is_encoder in (True,False):
        for decay in (True,False):
            group=[p for n,p in parameters if n.startswith('backbone.')==is_encoder and (p.ndim>=2)==decay]
            if group:groups.append({'params':group,'lr':cfg['encoder_lr'] if is_encoder else cfg['head_lr'],
                'base_lr':cfg['encoder_lr'] if is_encoder else cfg['head_lr'],'weight_decay':cfg.get('weight_decay',.01) if decay else 0.})
    optimizer=torch.optim.AdamW(groups,fused=device.type=='cuda');scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda')
    epochs=args.epochs or cfg['epochs'];accum=cfg.get('accumulation',1);planned=math.ceil(len(loader)/accum)*epochs
    support=np.bincount([data['docs'][i]['gold'] for i in dataset.rows],minlength=2).astype(float)
    weights=np.sqrt(support.sum()/np.maximum(support,1));weights=weights/weights.mean();weights=torch.tensor(weights,dtype=torch.float32,device=device)
    identity=digest({'config':cfg,'data':data['fingerprint'],'parent':file_hash(cfg['parent']),'epochs':epochs,'split':args.split,'world':world,
        'teacher':file_hash(args.teacher) if args.teacher else None,'features':signature(cfg) if args.features else None,
        'head_init':file_hash(cfg['head_init']) if cfg.get('head_init') else None,
        'backbone_init':file_hash(cfg['backbone_init']) if cfg.get('backbone_init') else None,'smoke':args.smoke})
    latest=root/'latest.pt';summary_path=root/'summary.json';best=root/'best.pt'
    if summary_path.exists():
        old=read_json(summary_path)
        if old['identity']!=identity:raise ValueError('Run configuration changed. Use a new PROJECT.')
        if old.get('complete') and best.exists():
            if rank==0:print('Already complete:',root,flush=True)
            return
    epoch=cursor=updates=skipped=bad=0;best_f1=-1.;best_epoch=0;history=[];elapsed_prior=0.
    if latest.exists():
        old=torch.load(latest,map_location='cpu',weights_only=False)
        if old['identity']!=identity:raise ValueError('Resume identity mismatch.')
        model.load_state_dict(old['model'],strict=not old['delta_only']);optimizer.load_state_dict(old['optimizer']);scaler.load_state_dict(old['scaler'])
        epoch,cursor,updates,skipped,bad=[old[k] for k in ('epoch','cursor','updates','skipped','bad')]
        best_f1,best_epoch,history,elapsed_prior=old['best_f1'],old['best_epoch'],old['history'],old['elapsed']
        restore_rng(old['rng'][rank],device)
    started=time.monotonic();trainable_snapshot={n for n,p in model.named_parameters() if p.requires_grad}
    delta_only=cfg.get('head_only') or cfg['method'] not in ('full','kd','gradual')
    def save(is_best=False,complete=False):
        states=[None]*world
        if world>1:dist.all_gather_object(states,rng_state(device))
        else:states[0]=rng_state(device)
        if rank==0:
            state=model.state_dict()
            if delta_only:state={k:v for k,v in state.items() if k in trainable_snapshot}
            light={'model':state,'delta_only':delta_only,'config':cfg,'provenance':provenance,'parameters':report,
                'data_fingerprint':data['fingerprint'],'training_origin':'own_phase1_phase2_phase3_phase4_cjpe',
                'parent_sha256':file_hash(cfg['parent']),
                'stage':args.split,'epoch':epoch,'updates':updates}
            if is_best:atomic_torch(best,light)
            checkpoint={**light,'identity':identity,'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),'rng':states,
                'cursor':cursor,'skipped':skipped,'bad':bad,'best_f1':best_f1,'best_epoch':best_epoch,'history':history,'elapsed':elapsed_prior+time.monotonic()-started}
            atomic_torch(latest,checkpoint);atomic_json(summary_path,{'identity':identity,'complete':complete,'best_epoch':best_epoch,
                'best_f1':best_f1,'epoch':epoch,'updates':updates,'amp_skipped_updates':skipped,'parameters':report,
                'elapsed_seconds':checkpoint['elapsed'],'history':history,'stop_reason':'smoke' if args.smoke else 'epochs_or_patience'})
        barrier()
    if rank==0:print(json.dumps({'experiment':cfg['name'],'stage':args.split,'head_only':cfg.get('head_only',False),
        'cases':len(dataset),'parameters':report,'effective_batch':cfg.get('batch_docs',1)*world*accum}),flush=True)
    while epoch<epochs:
        model.set_epoch(epoch);sampler.set_epoch(epoch);generator.manual_seed(seed+epoch)
        wrapped=DDP(model,device_ids=[device.index] if device.type=='cuda' else None,broadcast_buffers=False) if world>1 else model
        iterator=iter(loader)
        for _ in range(cursor):next(iterator)
        model.train();loss_sum=loss_count=0
        while cursor<len(loader):
            micro=[next(iterator) for _ in range(min(accum,len(loader)-cursor))];sample_count=sum(len(b['rows']) for b in micro);optimizer.zero_grad(set_to_none=True)
            ratio=(updates+1)/max(1,planned);warm=cfg.get('warmup_ratio',.08)
            factor=min(1.,ratio/max(warm,1e-6)) if ratio<warm else .1+.9*.5*(1+math.cos(math.pi*min(1.,(ratio-warm)/max(1-warm,1e-6))))
            for group in optimizer.param_groups:group['lr']=group['base_lr']*factor
            for index,batch in enumerate(micro):
                batch=to_device(batch,device);sync=wrapped.no_sync() if world>1 and index<len(micro)-1 else contextlib.nullcontext()
                with sync:
                    with amp(device):result=wrapped(batch);loss,components=classification_loss(result['logits'],batch,cfg,weights)
                    if not bool(torch.isfinite(loss)):raise FloatingPointError('Nonfinite loss; recovery states were retained.')
                    scaler.scale(loss*len(batch['rows'])/sample_count).backward()
                loss_sum+=float(loss.detach());loss_count+=1
            scaler.unscale_(optimizer);gradient=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.)
            scale=scaler.get_scale();scaler.step(optimizer);scaler.update();skipped+=int(scaler.get_scale()<scale);updates+=int(scaler.get_scale()>=scale);cursor+=len(micro)
            if rank==0 and updates and updates%cfg.get('log_every',50)==0:
                print(json.dumps({'epoch':epoch+1,'batch':cursor,'updates':updates,'loss':loss_sum/max(1,loss_count),
                    'gradient_norm':float(gradient),'elapsed_hours':(elapsed_prior+time.monotonic()-started)/3600,'amp_skipped_updates':skipped}),flush=True)
                loss_sum=loss_count=0
            if cursor<len(loader) and updates and updates%cfg.get('checkpoint_every',100)==0:save()
            if args.smoke:
                epoch+=1;cursor=0;best_epoch=epoch;best_f1=0.;save(True,True)
                if rank==0:print('PASS actual distributed update:',cfg['name'],flush=True)
                return
        epoch+=1;cursor=0
        if args.split=='train':
            values,_=infer_local(model,dev,rank,world,device,cfg.get('head_batch',16) if args.features else 1)
            values=gather_values(values,world);ordered=[data['docs'][i] for i in data['splits']['dev']]
            result=metric([d['gold'] for d in ordered],probabilities(torch.stack([values[d['id']] for d in ordered]).numpy()))
            score=result['macro_f1'];history.append({'epoch':epoch,**result})
            improved=score>best_f1+cfg.get('min_delta',0.)
            if improved:best_f1=score;best_epoch=epoch;bad=0
            else:bad+=1
            if rank==0:print(json.dumps({'epoch':epoch,'dev_macro_f1_at_half':score,'best_epoch':best_epoch}),flush=True)
        else:improved=True;best_epoch=epoch;best_f1=0.
        save(improved,epoch>=epochs or (args.split=='train' and bad>=cfg.get('patience',2)))
        del wrapped;gc.collect()
        if args.split=='train' and bad>=cfg.get('patience',2):break
    save(False,True)

def main():
    p=argparse.ArgumentParser();p.add_argument('--action',choices=['features','train','predict'],required=True)
    p.add_argument('--config');p.add_argument('--checkpoint');p.add_argument('--data',required=True);p.add_argument('--output',required=True)
    p.add_argument('--split',default='train',choices=['train','dev','test','all','every']);p.add_argument('--features');p.add_argument('--teacher')
    p.add_argument('--epochs',type=int);p.add_argument('--smoke',action='store_true');p.add_argument('--sections',default='all',choices=['all','first','last'])
    args=p.parse_args();rank,world,device=setup()
    cfg=read_json(args.config) if args.config else torch.load(args.checkpoint,map_location='cpu',weights_only=False)['config']
    try:
        if args.action=='features':feature_cache(args,cfg,rank,world,device)
        elif args.action=='predict':predict(args,cfg,rank,world,device)
        else:train(args,cfg,rank,world,device)
    finally:
        if dist.is_initialized():dist.destroy_process_group()
if __name__=='__main__':main()
'''
(CODE/'cj_run.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','cj_run.py')

In [ ]:
MODULE_SOURCE=r'''
"""Kaggle workflow: official data, seven real adaptation methods, optional lexical fusion and exports."""
import contextlib,gc,json,math,os,shutil,socket,subprocess,sys,time,zipfile
from pathlib import Path
import numpy as np
import torch
from cj_data import DATA_FILES,atomic_json,atomic_torch,digest,file_hash,read_json,load_records,metric,probabilities,choose_calibration,write_submission,Bags,collate,to_device
from cj_model import restore_phase3,build_model,parameter_report,classification_loss

def obtain_data(project,local_root=None,revision=None):
    destination=Path(project)/'data';destination.mkdir(parents=True,exist_ok=True);provenance=destination/'source.json'
    if local_root:
        source=Path(local_root)
        if (source/'Phase_4_CJPE').is_dir():source/='Phase_4_CJPE'
        missing=[name for name in DATA_FILES if not (source/name).is_file()]
        if missing:raise FileNotFoundError('Missing official Phase-4 files: '+', '.join(missing))
        origin={'kind':'attached_official_phase4','source':str(source)}
        for name in DATA_FILES:
            target=destination/name;target.parent.mkdir(parents=True,exist_ok=True)
            if target.resolve()!=(source/name).resolve():shutil.copy2(source/name,target)
    else:
        old=read_json(provenance) if provenance.exists() else None
        if old and old.get('kind')=='official_huggingface' and old.get('sha256') and (revision is None or revision==old.get('revision')):
            if all((destination/name).is_file() for name in DATA_FILES):
                hashes={name:file_hash(destination/name) for name in DATA_FILES}
                if hashes!=old['sha256']:raise ValueError('Cached official data changed; restore it or use a new PROJECT.')
                return str(destination)
        from huggingface_hub import HfApi,hf_hub_download
        token=os.environ.get('HF_TOKEN')
        if not token:
            try:
                from kaggle_secrets import UserSecretsClient
                token=UserSecretsClient().get_secret('HF_TOKEN')
            except Exception:token=None
        repo='Exploration-Lab/CS-781-Capstone';commit=old['revision'] if old and old.get('kind')=='official_huggingface' else revision
        try:
            commit=HfApi(token=token).dataset_info(repo,revision=commit).sha
            origin={'kind':'official_huggingface','repo':repo,'revision':commit};atomic_json(provenance,origin)
            for name in DATA_FILES:
                target=destination/name;target.parent.mkdir(parents=True,exist_ok=True)
                if not target.exists():
                    source=hf_hub_download(repo,'Phase_4_CJPE/'+name,repo_type='dataset',revision=commit,token=token)
                    shutil.copy2(source,target)
        except Exception as error:raise RuntimeError('Attach Phase_4_CJPE, or accept the official dataset terms and enable the HF_TOKEN Kaggle secret. Never paste API tokens into code.') from error
    origin['sha256']={name:file_hash(destination/name) for name in DATA_FILES};atomic_json(provenance,origin)
    return str(destination)

def experiments(parents,settings):
    common={'head_dim':settings['head_dim'],'dropout':.2,'encoder_dropout':.05,'chunk_batch':settings['chunk_batch'],
        'encoder_checkpointing':True,'head_only':False,'batch_docs':settings['batch_docs'],'head_batch':settings['head_batch'],
        'accumulation':settings['accumulation'],'encoder_lr':1e-5,'head_lr':2e-4,'weight_decay':.01,
        'epochs':settings['adapt_epochs'],'patience':settings['adapt_patience'],'min_delta':0.,'warmup_ratio':.08,
        'rank':8,'alpha':16.,'adapter_dim':48,'unfrozen_layers':2,'label_smoothing':.02,'seed':230437,
        'checkpoint_every':100,'log_every':50,'temperature':2.,'kd_alpha':.5,'view':'last' if settings.get('two_stage') else 'all'}
    specs=[('01_decoder_full','decoder','full','bigru'),('02_encoder_decoder_partial','encoder_decoder','partial','transformer'),
        ('03_local_gqa_lorma','local_gqa','lorma','bigru'),('04_decoder_adapter','decoder','adapter','bigru'),
        ('05_hybrid_bitfit','hybrid','bitfit','bigru'),('06_encoder_decoder_gradual','encoder_decoder','gradual','bigru'),
        ('07_decoder_student_kd','decoder','kd','bigru')]
    rows=[]
    for name,architecture,method,head in specs:
        cfg={**common,'name':name,'architecture':architecture,'method':method,'head_type':head,'parent':parents[architecture]}
        if method in ('lorma','adapter','bitfit'):cfg['encoder_lr']=1e-4
        if method=='kd':cfg['student_layers']=settings['student_layers']
        rows.append(cfg)
    return rows
def launch(action,args,project,gpus=2,log=None):
    script=Path(project)/'code/cj_run.py'
    if gpus==1:cmd=[sys.executable,'-u',str(script),'--action',action,*map(str,args)]
    else:
        with socket.socket() as sock:sock.bind(('127.0.0.1',0));port=sock.getsockname()[1]
        cmd=[sys.executable,'-m','torch.distributed.run','--nnodes=1','--master_addr=127.0.0.1',f'--master_port={port}',
            f'--nproc_per_node={gpus}',str(script),'--action',action,*map(str,args)]
    env=dict(os.environ);env['PYTHONPATH']=str(Path(project)/'code')+os.pathsep+env.get('PYTHONPATH','')
    env['TOKENIZERS_PARALLELISM']='false';env['OMP_NUM_THREADS']='1'
    handle=Path(log).open('a',encoding='utf-8') if log else None
    try:
        process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        for line in process.stdout:
            print(line,end='',flush=True)
            if handle:handle.write(line);handle.flush()
        if process.wait()!=0:raise RuntimeError(f'{action} failed. Inspect {log or "cell output"}. Earlier candidates and resume states remain available.')
    finally:
        if handle:handle.close()
def frozen_key(cfg):return cfg['architecture']+('_student_'+str(cfg['student_layers']) if cfg['method']=='kd' else '')
def warm_config(cfg,settings):return {**cfg,'head_only':True,'view':'all','head_lr':6e-4,'batch_docs':settings['head_batch'],'accumulation':1,
    'epochs':settings['head_epochs'],'patience':settings['head_patience']}

def lexical_branch(project,data_root,data_path,enabled=True):
    if not enabled:return None
    import joblib
    from scipy.sparse import hstack
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    project=Path(project);root=project/'lexical';root.mkdir(parents=True,exist_ok=True);data=read_json(data_path);result_path=root/'probabilities.pt'
    identity=digest({'data':data['fingerprint'],'feature':'word_1_2_full_and_last2048words','version':1})
    if result_path.exists():
        result=torch.load(result_path,map_location='cpu',weights_only=False)
        if result['identity']!=identity:raise ValueError('Lexical model data changed.')
        return result
    rows=load_records(data_root);byid={r['id']:r for r in rows};ids=[d['id'] for d in data['docs']]
    texts=[byid[did]['text'] for did in ids];tails=[' '.join(t.split()[-2048:]) for t in texts];train=data['splits']['train']
    vectorizer=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=150_000,sublinear_tf=True,dtype=np.float32)
    start=time.monotonic()
    try:vectorizer.fit([texts[i] for i in train])
    except ValueError as error:raise ValueError('Lexical vocabulary is empty. Disable USE_LEXICAL_FUSION for tiny/non-word smoke data.') from error
    matrix=hstack((vectorizer.transform(texts),vectorizer.transform(tails)),format='csr')
    labels=[data['docs'][i]['gold'] for i in train]
    classifier=LogisticRegression(C=2.,class_weight='balanced',solver='liblinear',max_iter=500,random_state=230437)
    classifier.fit(matrix[train],labels);p=classifier.predict_proba(matrix)[:,1].astype(np.float32)
    values={did:float(value) for did,value in zip(ids,p)};dev=data['splits']['dev']
    scores=metric([data['docs'][i]['gold'] for i in dev],p[dev])
    joblib.dump({'vectorizer':vectorizer,'classifier':classifier,'identity':identity},root/'model.joblib',compress=3)
    result={'identity':identity,'values':values,'data_fingerprint':data['fingerprint'],'dev_metrics':scores,
        'elapsed_seconds':time.monotonic()-start,'fit_split':'train_only','feature':'full_text_and_final2048_words'}
    atomic_torch(result_path,result);atomic_json(root/'metrics.json',{k:v for k,v in result.items() if k!='values'})
    print('Train-only lexical branch:',scores,'seconds:',round(result['elapsed_seconds'],1),flush=True)
    return result

def benchmark(rows,data_path,device=None):
    device=device or torch.device('cuda',0);data=read_json(data_path);dataset=Bags(data_path,'train');measurements=[]
    if device.type=='cuda':torch.cuda.set_device(device)
    for cfg in rows:
        dataset=Bags(data_path,'train',view=cfg.get('view','all'))
        torch.manual_seed(cfg['seed']);model,_=build_model(cfg);model.to(device).train();model.set_epoch(0)
        row=dataset[0];row['chunks']=row['chunks'][:4];row['position']=row['position'][:4]
        batch=to_device(collate([row]),device)
        if cfg['method']=='kd':batch['teacher']=torch.zeros(1,2,device=device)
        optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=cfg['head_lr'],fused=device.type=='cuda')
        scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda');times=[];peak=0
        for iteration in range(3):
            if device.type=='cuda':torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()
            start=time.monotonic();optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16) if device.type=='cuda' else contextlib.nullcontext():
                result=model(batch);loss,_=classification_loss(result['logits'],batch,cfg,torch.ones(2,device=device))
            scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
            if device.type=='cuda':torch.cuda.synchronize();peak=max(peak,torch.cuda.max_memory_reserved()/1024**3)
            if iteration:times.append(time.monotonic()-start)
        n=len(row['chunks']);seconds=sum(times)/len(times);total_chunks=sum(len(data['docs'][i]['chunks']) if cfg.get('view','all')=='all' else 1 for i in data['splits']['train'])
        result={'name':cfg['name'],'timed_chunks':n,'seconds_per_sample':seconds,'seconds_per_chunk':seconds/n,
            'peak_reserved_gib':peak,'parameters':parameter_report(model),'rough_adaptation_hours':total_chunks/n*seconds*cfg['epochs']/2/3600,
            'forecast_excludes':'feature extraction, heads, validation, KD caches, DDP, loading and I/O; variable-length bags differ'}
        measurements.append(result);print(json.dumps(result),flush=True)
        del model,optimizer,scaler,batch,result,loss;gc.collect()
        if device.type=='cuda':torch.cuda.empty_cache()
    return measurements
def smoke_all(project,rows,data_path,settings):
    project=Path(project);data=read_json(data_path);root=project/'preflight';root.mkdir(parents=True,exist_ok=True)
    teacher=root/'synthetic_teacher_for_smoke_only.pt'
    atomic_torch(teacher,{'data_fingerprint':data['fingerprint'],'split':'train','teacher_parameters':10**12,'smoke_only':True,
        'values':{data['docs'][i]['id']:torch.zeros(2) for i in data['splits']['train']}})
    for cfg in rows:
        current={**cfg,'batch_docs':1,'accumulation':1,'epochs':1,'checkpoint_every':10**9};path=root/f"{cfg['name']}.json";atomic_json(path,current)
        destination=root/cfg['name'];args=['--config',path,'--data',data_path,'--output',destination,'--smoke']
        if cfg['method']=='kd':args+=['--teacher',teacher]
        launch('train',args,project,settings['gpus'],root/f"{cfg['name']}.log")
        if not read_json(destination/'summary.json')['complete']:raise ValueError('Incomplete GPU smoke test.')
        shutil.rmtree(destination)
    teacher.unlink();atomic_json(root/'passed.json',{'experiments':[r['name'] for r in rows],'data_fingerprint':data['fingerprint'],'world_size':settings['gpus']})

def _predict(project,checkpoint,data_path,split,output,settings,features=None,sections='all'):
    args=['--checkpoint',checkpoint,'--data',data_path,'--split',split,'--output',output,'--sections',sections]
    if features:args+=['--features',features]
    launch('predict',args,project,settings['gpus'],Path(output).parent/(Path(output).name+'.log'))
    return torch.load(Path(output)/'complete.pt',map_location='cpu',weights_only=False)
def _array(raw,data,split):return probabilities(torch.stack([raw['values'][data['docs'][i]['id']] for i in data['splits'][split]]).numpy())
def _fuse(neural,ids,lexical,calibration):
    weight=calibration['lexical_weight']
    return np.asarray(neural)*(1-weight)+np.asarray([lexical['values'][did] for did in ids])*weight if weight else np.asarray(neural)
def _teacher_cache(project,item,data_path,split,root,settings,lexical):
    data=read_json(data_path);root=Path(root);root.mkdir(parents=True,exist_ok=True);target=root/'teacher.pt'
    identity=digest({'source':file_hash(item['selection_checkpoint']),'calibration':item['calibration'],'data':data['fingerprint'],
        'split':split,'lexical_identity':lexical['identity'] if lexical else None})
    if target.exists():
        saved=torch.load(target,map_location='cpu',weights_only=False)
        if saved['identity']!=identity:raise ValueError('Teacher changed; use a new PROJECT.')
        return target
    raw=_predict(project,item['selection_checkpoint'],data_path,split,root/'raw',settings,item.get('selected_features'))
    ids=[data['docs'][i]['id'] for i in data['splits'][split]];p=_fuse(_array(raw,data,split),ids,lexical,item['calibration'])
    p=np.clip(p,1e-6,1-1e-6);values={did:torch.tensor([math.log(1-q),math.log(q)],dtype=torch.float32) for did,q in zip(ids,p)}
    atomic_torch(target,{'identity':identity,'data_fingerprint':data['fingerprint'],'split':split,'values':values,
        'teacher_parameters':item['parameters']['total'],'teacher_checkpoint_sha256':file_hash(item['selection_checkpoint']),
        'teacher_name':item['name'],'teacher_kind':'selected_phase4_classifier_with_optional_train_only_lexical_branch'})
    return target

def run_all(project,rows,data_path,settings,lexical=None):
    project=Path(project);data=read_json(data_path);catalog=[];dev_ids=[data['docs'][i]['id'] for i in data['splits']['dev']]
    test_ids=[data['docs'][i]['id'] for i in data['splits']['test']];truth=[data['docs'][i]['gold'] for i in data['splits']['dev']]
    atomic_json(project/'dev_truth.json',[{'id':did,'gold':gold} for did,gold in zip(dev_ids,truth)])
    for original in rows:
        cfg=dict(original);root=project/'runs'/cfg['name'];root.mkdir(parents=True,exist_ok=True);teacher=None
        if cfg['method']=='kd':
            strongest=max(catalog,key=lambda item:item['dev_metrics']['macro_f1'])
            teacher=_teacher_cache(project,strongest,data_path,'train',root/'teacher_train',settings,lexical)
            cfg['teacher_name']=strongest['name']
        feature_root=project/'features'/frozen_key(cfg);feature_root.mkdir(parents=True,exist_ok=True)
        feature_cfg={**cfg,'head_only':True,'method':'kd' if cfg['method']=='kd' else 'full'}
        feature_path=feature_root/'config.json';atomic_json(feature_path,feature_cfg)
        launch('features',['--config',feature_path,'--data',data_path,'--output',feature_root],project,settings['gpus'],feature_root/'features.log')
        warm=warm_config(cfg,settings);warm_path=root/'warm_config.json';atomic_json(warm_path,warm)
        args=['--config',warm_path,'--data',data_path,'--split','train','--output',root/'warm_selection','--features',feature_root]
        if teacher:args+=['--teacher',teacher]
        launch('train',args,project,1,root/'warm_selection.log')
        cfg['head_init']=str(root/'warm_selection/best.pt');cfg_path=root/'config.json';atomic_json(cfg_path,cfg)
        args=['--config',cfg_path,'--data',data_path,'--split','train','--output',root/'selection']
        if teacher:args+=['--teacher',teacher]
        launch('train',args,project,settings['gpus'],root/'selection.log')
        adapted_checkpoint=root/'selection/best.pt';adapted_features=None;hierarchy_epoch=None
        if settings.get('two_stage'):
            hierarchy={**warm_config(cfg,settings),'backbone_init':str(adapted_checkpoint),'head_init':str(adapted_checkpoint)}
            hierarchy_path=root/'hierarchy_config.json';atomic_json(hierarchy_path,hierarchy)
            adapted_features=root/'adapted_features';adapted_features.mkdir(exist_ok=True)
            launch('features',['--config',hierarchy_path,'--data',data_path,'--output',adapted_features],project,settings['gpus'],root/'adapted_features.log')
            args=['--config',hierarchy_path,'--data',data_path,'--split','train','--output',root/'hierarchy_selection','--features',adapted_features]
            if teacher:args+=['--teacher',teacher]
            launch('train',args,project,1,root/'hierarchy_selection.log')
            adapted_checkpoint=root/'hierarchy_selection/best.pt';hierarchy_epoch=read_json(root/'hierarchy_selection/summary.json')['best_epoch']
        warm_raw=_predict(project,root/'warm_selection/best.pt',data_path,'dev',root/'warm_dev',settings,feature_root)
        adapted_raw=_predict(project,adapted_checkpoint,data_path,'dev',root/'adapted_dev',settings,adapted_features)
        lexical_dev=np.asarray([lexical['values'][did] for did in dev_ids]) if lexical else None
        warm_metrics,warm_cal=choose_calibration(truth,_array(warm_raw,data,'dev'),lexical_dev)
        adapt_metrics,adapt_cal=choose_calibration(truth,_array(adapted_raw,data,'dev'),lexical_dev)
        # Preserve the stronger held-out model instead of forcing a degraded adaptation into a ZIP.
        use_warm=settings.get('allow_warm_fallback',True) and warm_metrics['macro_f1']>adapt_metrics['macro_f1']+1e-9
        stage='warmup' if use_warm else 'hierarchy' if settings.get('two_stage') else 'adaptation'
        selected=root/'warm_selection/best.pt' if use_warm else adapted_checkpoint
        selected_features=feature_root if use_warm else adapted_features
        scores,calibration=(warm_metrics,warm_cal) if use_warm else (adapt_metrics,adapt_cal);chosen=warm_raw if use_warm else adapted_raw
        summary=read_json(selected.parent/'summary.json')
        dev_probability=_fuse(_array(chosen,data,'dev'),dev_ids,lexical,calibration)
        dev_prediction=[{'id':did,'pred':int(q>=calibration['threshold'])} for did,q in zip(dev_ids,dev_probability)]
        atomic_json(root/'dev_prediction.json',dev_prediction);atomic_json(root/'calibration.json',calibration)
        atomic_json(root/'dev_metrics.json',{'selected_stage':stage,'selected':scores,'warmup':warm_metrics,'adaptation':adapt_metrics})
        raw=_predict(project,selected,data_path,'test',root/'selected_test',settings,selected_features)
        p=_fuse(_array(raw,data,'test'),test_ids,lexical,calibration);preds=[int(q>=calibration['threshold']) for q in p]
        submission=project/'submissions'/f"{cfg['name']}.zip";prediction,sha=write_submission(test_ids,preds,submission)
        atomic_json(root/'prediction.json',prediction)
        item={'name':cfg['name'],'architecture':cfg['architecture'],'method':cfg['method'],'selected_stage':stage,
            'dev_metrics':scores,'warmup_metrics':warm_metrics,'adaptation_metrics':adapt_metrics,'calibration':calibration,
            'parameters':summary['parameters'],'selection_checkpoint':str(selected),'test_checkpoint':str(selected),
            'features':str(feature_root),'submission':str(submission),'prediction_sha256':sha,'dev_used_for_selection':True,
            'selected_features':str(selected_features) if selected_features else None,'hierarchy_epoch':hierarchy_epoch,
            'encoder_stage_parameters':read_json(root/'selection/summary.json')['parameters'],
            'warmup_epoch':read_json(root/'warm_selection/summary.json')['best_epoch'],'adaptation_epoch':read_json(root/'selection/summary.json')['best_epoch'],
            'teacher_name':cfg.get('teacher_name')}
        catalog.append(item);atomic_json(project/'catalog.json',catalog)
        print('READY:',submission,'development-tuned Macro-F1:',round(scores['macro_f1'],5),'selected:',stage,flush=True)
    if len(catalog)!=7 or len({r['architecture'] for r in catalog})!=4:raise ValueError('Experiment coverage incomplete.')
    distinct=len({r['prediction_sha256'] for r in catalog})
    print('Seven experiment ZIPs; distinct prediction sets:',distinct,flush=True)
    if distinct<7:print('Identical predictions do not count as different submissions. Inspect duplicate candidates; labels have not been altered to manufacture uniqueness.',flush=True)
    if any(r['selected_stage']=='warmup' for r in catalog):print('Some selected ZIPs use a frozen warmup fallback. Their adaptation experiments ran, but check method diversity among the actual selected submissions.',flush=True)
    return catalog

def bonus_ensemble(project,data_path,lexical=None):
    """Optional extra ZIP only for a development improvement; never a claimed eighth adaptation method."""
    project=Path(project);data=read_json(data_path);catalog=read_json(project/'catalog.json');ranked=sorted(catalog,key=lambda r:r['dev_metrics']['macro_f1'],reverse=True)
    dev_ids=[data['docs'][i]['id'] for i in data['splits']['dev']];test_ids=[data['docs'][i]['id'] for i in data['splits']['test']]
    truth=[data['docs'][i]['gold'] for i in data['splits']['dev']];groups=[ranked[:3]];families=[];seen=set()
    for item in ranked:
        if item['architecture'] not in seen:families.append(item);seen.add(item['architecture'])
    groups.append(families);best_score=ranked[0]['dev_metrics']['macro_f1'];winner=None
    for members in groups:
        dev_prob=[];test_prob=[]
        for item in members:
            root=project/'runs'/item['name']
            dev_dir='joint_dev' if item['selected_stage']=='joint' else 'warm_dev' if item['selected_stage']=='warmup' else 'adapted_dev'
            test_dir='refit_test' if item.get('all_label_refit') else 'joint_test' if item['selected_stage']=='joint' else 'selected_test'
            dev_raw=torch.load(root/dev_dir/'complete.pt',map_location='cpu',weights_only=False)
            test_raw=torch.load(root/test_dir/'complete.pt',map_location='cpu',weights_only=False)
            adjust=lambda p:1/(1+np.exp(-np.clip(np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))-math.log(item['calibration']['threshold']/(1-item['calibration']['threshold'])),-30,30)))
            dev_prob.append(adjust(_fuse(_array(dev_raw,data,'dev'),dev_ids,lexical,item['calibration'])))
            test_prob.append(adjust(_fuse(_array(test_raw,data,'test'),test_ids,lexical,item['calibration'])))
        scores,cal=choose_calibration(truth,np.mean(dev_prob,axis=0))
        if scores['macro_f1']>best_score+1e-9:
            best_score=scores['macro_f1'];winner=(members,scores,cal,np.mean(test_prob,axis=0))
    if winner is None:
        atomic_json(project/'recommended_submission.json',{'submission':ranked[0]['submission'],'name':ranked[0]['name'],'dev_metrics':ranked[0]['dev_metrics'],'kind':'single_selected_classifier'})
        print('No measured ensemble gain. Recommended existing ZIP:',ranked[0]['submission'],flush=True);return None
    members,scores,cal,p=winner;path=project/'bonus/score_ensemble.zip'
    prediction,sha=write_submission(test_ids,[int(q>=cal['threshold']) for q in p],path)
    atomic_json(project/'bonus/prediction.json',prediction)
    record={'submission':str(path),'kind':'development_selected_ensemble','members':[r['name'] for r in members],
        'weights':[1/len(members)]*len(members),'dev_metrics':scores,'calibration':cal,'prediction_sha256':sha,
        'counts_as_new_adaptation_method':False,'dev_used_for_selection':True}
    atomic_json(project/'recommended_submission.json',record);print('Optional score ensemble:',path,scores,flush=True);return record

def export_manifest(project):
    project=Path(project).resolve();catalog=read_json(project/'catalog.json');models=[]
    for item in catalog:
        checkpoint=Path(item['test_checkpoint']).resolve();saved=torch.load(checkpoint,map_location='cpu',weights_only=False)
        dependencies={}
        for key in ('parent','backbone_init'):
            if not saved['config'].get(key):continue
            path=Path(saved['config'][key]).resolve()
            if not path.is_relative_to(project):raise ValueError('Portable dependency must be inside PROJECT.')
            dependencies[key]={'relative_path':str(path.relative_to(project)),'sha256':file_hash(path)}
        models.append({'name':item['name'],'architecture':item['architecture'],'method':item['method'],
            'checkpoint':str(checkpoint.relative_to(project)),'checkpoint_sha256':file_hash(checkpoint),
            'dependencies':dependencies,'config':saved['config'],'provenance':saved['provenance']})
    atomic_json(project/'phase4_model_manifest.json',{'format_version':1,'tokenizer':'parents/tokenizer.json',
        'tokenizer_sha256':file_hash(project/'parents/tokenizer.json'),'models':models,
        'restoration':'Resolve checkpoint and dependency relative_path against the extracted capstone_phase4 root; override config paths before building.'})
    return project/'phase4_model_manifest.json'

def joint_refine_top(project,rows,data_path,settings,lexical,top_k=1):
    project=Path(project);catalog=read_json(project/'catalog.json');data=read_json(data_path);byname={r['name']:r for r in rows}
    truth=[data['docs'][i]['gold'] for i in data['splits']['dev']];dev_ids=[data['docs'][i]['id'] for i in data['splits']['dev']]
    test_ids=[data['docs'][i]['id'] for i in data['splits']['test']]
    for item in sorted(catalog,key=lambda r:r['dev_metrics']['macro_f1'],reverse=True)[:top_k]:
        if item.get('joint_refined'):continue
        root=project/'runs'/item['name'];selected=torch.load(item['selection_checkpoint'],map_location='cpu',weights_only=False)['config']
        cfg={**byname[item['name']],'view':'all','head_only':False,'head_init':item['selection_checkpoint'],
            'epochs':settings.get('joint_epochs',2),'patience':settings.get('joint_patience',1),
            'batch_docs':1,'accumulation':settings.get('joint_accumulation',4)}
        if selected.get('backbone_init'):cfg['backbone_init']=selected['backbone_init']
        path=root/'joint_config.json';atomic_json(path,cfg)
        args=['--config',path,'--data',data_path,'--split','train','--output',root/'joint_selection']
        if cfg['method']=='kd':args+=['--teacher',root/'teacher_train/teacher.pt']
        launch('train',args,project,settings['gpus'],root/'joint_selection.log');checkpoint=root/'joint_selection/best.pt'
        raw=_predict(project,checkpoint,data_path,'dev',root/'joint_dev',settings)
        scores,cal=choose_calibration(truth,_array(raw,data,'dev'),np.asarray([lexical['values'][did] for did in dev_ids]) if lexical else None)
        item['joint_refined']=True;item['joint_dev_metrics']=scores
        atomic_json(root/'joint_comparison.json',{'before':item['dev_metrics'],'joint':scores,'accepted':scores['macro_f1']>item['dev_metrics']['macro_f1']+1e-9})
        if scores['macro_f1']>item['dev_metrics']['macro_f1']+1e-9:
            raw_test=_predict(project,checkpoint,data_path,'test',root/'joint_test',settings)
            p=_fuse(_array(raw_test,data,'test'),test_ids,lexical,cal)
            backup=root/'before_joint_submission.zip'
            if not backup.exists():shutil.copy2(item['submission'],backup)
            prediction,sha=write_submission(test_ids,[int(q>=cal['threshold']) for q in p],item['submission'])
            atomic_json(root/'prediction.json',prediction)
            pdev=_fuse(_array(raw,data,'dev'),dev_ids,lexical,cal)
            atomic_json(root/'dev_prediction.json',[{'id':did,'pred':int(q>=cal['threshold'])} for did,q in zip(dev_ids,pdev)])
            item.update(pre_joint_checkpoint=item['selection_checkpoint'],selection_checkpoint=str(checkpoint),test_checkpoint=str(checkpoint),
                selected_stage='joint',selected_features=None,calibration=cal,dev_metrics=scores,prediction_sha256=sha,
                parameters=read_json(root/'joint_selection/summary.json')['parameters'],
                joint_epoch=read_json(root/'joint_selection/summary.json')['best_epoch'])
            atomic_json(root/'calibration.json',cal)
        atomic_json(project/'catalog.json',catalog)
        print('Joint refinement:',item['name'],'old-vs-joint comparison saved; selected stage:',item['selected_stage'],flush=True)
    return catalog

def refit_top(project,rows,data_path,settings,lexical,top_k=1):
    """Optional fixed-epoch all-label refit after all seven held-out submissions already exist."""
    if not top_k:return read_json(Path(project)/'catalog.json')
    project=Path(project);data=read_json(data_path);catalog=read_json(project/'catalog.json');original_byname={r['name']:r for r in rows}
    ids=[data['docs'][i]['id'] for i in data['splits']['test']];selected=sorted(catalog,key=lambda r:r['dev_metrics']['macro_f1'],reverse=True)[:top_k]
    for item in selected:
        if item.get('all_label_refit'):continue
        cfg=dict(original_byname[item['name']]);root=project/'runs'/cfg['name'];teacher=None
        if cfg['method']=='kd':
            candidates=[r for r in catalog if r['method']!='kd'];strongest=max(candidates,key=lambda r:r['dev_metrics']['macro_f1'])
            teacher=_teacher_cache(project,strongest,data_path,'all',root/'teacher_all',settings,lexical)
        warm=warm_config(cfg,settings);path=root/'warm_refit_config.json';atomic_json(path,warm)
        args=['--config',path,'--data',data_path,'--split','all','--output',root/'warm_refit','--features',item['features'],'--epochs',max(1,item['warmup_epoch'])]
        if teacher:args+=['--teacher',teacher]
        launch('train',args,project,1,root/'warm_refit.log');checkpoint=root/'warm_refit/best.pt'
        features=item['features']
        if item['selected_stage'] in ('adaptation','hierarchy','joint'):
            cfg['head_init']=str(checkpoint);path=root/'refit_config.json';atomic_json(path,cfg)
            args=['--config',path,'--data',data_path,'--split','all','--output',root/'refit','--epochs',max(1,item['adaptation_epoch'])]
            if teacher:args+=['--teacher',teacher]
            launch('train',args,project,settings['gpus'],root/'refit.log');checkpoint=root/'refit/best.pt';features=None
            if settings.get('two_stage'):
                hierarchy={**warm_config(cfg,settings),'backbone_init':str(checkpoint),'head_init':str(checkpoint)}
                path=root/'hierarchy_refit_config.json';atomic_json(path,hierarchy);features=root/'refit_features';features.mkdir(exist_ok=True)
                launch('features',['--config',path,'--data',data_path,'--output',features],project,settings['gpus'],root/'refit_features.log')
                args=['--config',path,'--data',data_path,'--split','all','--output',root/'hierarchy_refit','--features',features,
                    '--epochs',max(1,item.get('hierarchy_epoch') or item['warmup_epoch'])]
                if teacher:args+=['--teacher',teacher]
                launch('train',args,project,1,root/'hierarchy_refit.log');checkpoint=root/'hierarchy_refit/best.pt'
            if item['selected_stage']=='joint':
                joint={**cfg,'view':'all','head_init':str(checkpoint),'epochs':settings.get('joint_epochs',2),'batch_docs':1,
                    'accumulation':settings.get('joint_accumulation',4)}
                if settings.get('two_stage'):joint['backbone_init']=str(root/'refit/best.pt')
                path=root/'joint_refit_config.json';atomic_json(path,joint)
                args=['--config',path,'--data',data_path,'--split','all','--output',root/'joint_refit','--epochs',max(1,item['joint_epoch'])]
                if teacher:args+=['--teacher',teacher]
                launch('train',args,project,settings['gpus'],root/'joint_refit.log');checkpoint=root/'joint_refit/best.pt';features=None
        raw=_predict(project,checkpoint,data_path,'test',root/'refit_test',settings,features)
        p=_fuse(_array(raw,data,'test'),ids,lexical,item['calibration']);preds=[int(q>=item['calibration']['threshold']) for q in p]
        # Keep the held-out-model ZIP as a recovery artifact outside submissions/.
        recovery=root/'selection_submission.zip'
        if not recovery.exists():shutil.copy2(item['submission'],recovery)
        prediction,sha=write_submission(ids,preds,item['submission']);atomic_json(root/'prediction.json',prediction)
        item.update(test_checkpoint=str(checkpoint),prediction_sha256=sha,all_label_refit=True)
        atomic_json(project/'catalog.json',catalog)
    return catalog
def section_audit(project,data_path,settings):
    project=Path(project);catalog=read_json(project/'catalog.json');best=max(catalog,key=lambda r:r['dev_metrics']['macro_f1']);root=project/'section_audit';root.mkdir(exist_ok=True)
    data=read_json(data_path);y=[data['docs'][i]['gold'] for i in data['splits']['dev']];results={}
    for section in ('all','first','last'):
        raw=_predict(project,best['selection_checkpoint'],data_path,'dev',root/section,settings,
            best.get('selected_features'),section)
        results[section]=metric(y,_array(raw,data,'dev'))
    atomic_json(root/'metrics.json',{'model':best['name'],'scores':results,'interpretation':'Post-training occlusion sensitivity, not separately trained section-only models; attention is not causal explanation.'})
    return results
def bundle_working(working,destination):
    working=Path(working).resolve();destination=Path(destination).resolve();temporary=destination.with_suffix('.tmp')
    with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
        for source in working.rglob('*'):
            if source.is_file() and source.resolve() not in (destination,temporary):archive.write(source,source.relative_to(working))
    temporary.replace(destination);return str(destination)
'''
(CODE/'cj_workflow.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','cj_workflow.py')

In [ ]:
MODULE_SOURCE=r'''
"""Read supplied test truth only after predictions are frozen. No training consumer imports this."""
from pathlib import Path
import json,zipfile
from cj_data import records,read_json,atomic_json,metric

def evaluate_test_labels(root,submission,output=None):
    # Using the labeled-record parser here validates labels without putting them into model data.
    truth=records(read_json(Path(root)/'Test/test.json'),'dev')
    submission=Path(submission)
    if submission.suffix=='.zip':
        with zipfile.ZipFile(submission) as archive:predictions=json.loads(archive.read('prediction.json'))
    else:predictions=read_json(submission)
    if not isinstance(predictions,list) or not predictions:raise ValueError('Predictions must be a nonempty list.')
    pred={}
    for row in predictions:
        did=row.get('id');value=row.get('pred')
        if not isinstance(did,str) or not did or did in pred or type(value) is not int or value not in (0,1):
            raise ValueError('Invalid or repeated prediction ID/label.')
        pred[did]=value
    labels={}
    for row in truth:labels.setdefault(row['source_id'],set()).add(row['gold'])
    if set(pred)!=set(labels):raise ValueError('Test truth and prediction source-case IDs differ.')
    conflicts=[did for did,values in labels.items() if len(values)>1]
    record_scores=metric([row['gold'] for row in truth],[pred[row['source_id']] for row in truth])
    case_scores=None
    if not conflicts:case_scores=metric([next(iter(values)) for values in labels.values()],[pred[did] for did in labels])
    result={'evaluation_only':True,'submission':str(submission),'case_count':len(labels),'distinct_labeled_records':len(truth),
        'case_metrics':case_scores,'distinct_record_diagnostic':record_scores,
        'cases_with_conflicting_labels':len(conflicts),'conflicting_case_id_examples':conflicts[:20],
        'note':'No test-based checkpoint, threshold, fusion, ensemble or submission selection. Case metrics require unambiguous truth per original ID.'}
    if output is not None:atomic_json(output,result)
    return result
'''
(CODE/'cj_test_eval.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','cj_test_eval.py')

In [ ]:
MODULE_SOURCE=r'''
#!/usr/bin/env python3
"""Local evaluator for Phase 4.

Truth JSON: a list of objects, [{"id":"...", "gold":0|1}, ...]
Prediction JSON: [{"id":"...", "pred":0|1}, ...]
"""
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out

def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 4 binary predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)
'''
(CODE/'evaluate_phase4_official.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','evaluate_phase4_official.py')

In [ ]:
importlib.invalidate_caches()
for name in ('cj_workflow','cj_run','cj_model','cj_data','legal_models'):sys.modules.pop(name,None)
from cj_workflow import (restore_phase3,obtain_data,experiments,benchmark,smoke_all,lexical_branch,run_all,
                         joint_refine_top,refit_top,section_audit,bonus_ensemble,export_manifest,bundle_working)
from cj_data import prepare_data,read_json,atomic_json,digest
design={'settings':SETTINGS,'lexical_fusion':USE_LEXICAL_FUSION}
design_path=PROJECT/'run_design.json'
if design_path.exists() and read_json(design_path)!=design:
    raise ValueError('The training design changed. Restore the previous SETTINGS or choose a new PROJECT.')
atomic_json(design_path,design)
parents,tokenizer_path=restore_phase3(PHASE3_ROOT,PROJECT/'parents',ALLOW_SELECTION_FALLBACK,PARENT_OVERRIDES)
print('Selected Phase-3 models:')
print(json.dumps(read_json(PROJECT/'parents/input_manifest.json'),indent=2))
print('Unchanged tokenizer:',tokenizer_path)

## Official data and audit

The parser validates record/column layouts and integer labels before training. It preserves original IDs, discards supplied test labels from model-facing records, keeps all tokens and checks split overlap. The token cache and feature banks are disk-backed. The reported full-document feature size is a storage estimate; it is not a runtime estimate.

Training can contain repeated source case IDs. Exact ID/text/label copies are collapsed; distinct instances receive internal training IDs while their original case ID is retained as `source_id`. Training instances of held-out case IDs are excluded. Distinct dev records keep their labels and receive internal IDs when needed. Test texts are grouped by original case ID, with one prediction per case; see nested dev/test audits. Dev source cases occurring in test are excluded from selection. The `training_id_resolution` audit reports these decisions. Existing sessions using the original notebook can run `Phase4_Duplicate_ID_Fix_Cell.py` before this cell without downloading the data again.

In [ ]:
data_root=obtain_data(PROJECT,DATA_ROOT,DATA_REVISION)
DATA_PATH=prepare_data(data_root,tokenizer_path,PROJECT/'token_cache',SETTINGS['chunk_tokens'],SETTINGS['overlap'])
data=read_json(DATA_PATH)
assert all(data['docs'][i]['gold'] is None for i in data['splits']['test'])
assert set(data['splits']['all'])==set(data['splits']['train'])|set(data['splits']['dev'])
assert not set(data['splits']['all'])&set(data['splits']['test'])
print('PASS: test targets excluded; training/refit splits contain no test records.')
print(json.dumps(data['audit'],indent=2))
print('Original tokens:',data['total_tokens'],'Complete document windows:',data['total_chunks'])
atomic_json(PROJECT/'data_audit.json',data['audit'])
from tokenizers import Tokenizer
vocab_size=Tokenizer.from_file(str(tokenizer_path)).get_vocab_size()
parent_dimensions=[]
for architecture,path in parents.items():
    checkpoint=torch.load(path,map_location='cpu',weights_only=False)
    assert vocab_size<=checkpoint['config']['vocab_size'],'Tokenizer vocabulary exceeds backbone embedding size.'
    assert SETTINGS['chunk_tokens']<=checkpoint['config']['max_positions'],'Chunk size exceeds trained position buffer.'
    parent_dimensions.append(checkpoint['config']['dim'])
    del checkpoint
banks=12 if SETTINGS['two_stage'] else 5
feature_gib=data['total_chunks']*2*max(parent_dimensions)*2*banks/1024**3
print('Conservative complete-feature-bank estimate:',round(feature_gib,2),'GiB, before checkpoints/archive.')
print('Working disk free:',round(shutil.disk_usage(WORKING).free/1024**3,2),'GiB')

## Seven experiments and hardware preflight

Methods are full fine-tuning, fixed partial tuning, LoRMA+, adapters, BitFit, gradual unfreezing and knowledge distillation. All four Phase-3 families participate. Candidate 02 uses a chunk Transformer; the others use native BiGRU context.

The benchmark runs real forward/backward/optimizer updates on GPU 0, allocating optimizer state. Its forecast covers encoder adaptation only and excludes feature extraction, head epochs, validation, teacher inference, DDP and I/O. The next test runs an actual two-process update per method. Zero teacher logits are allowed only for benchmark/preflight; the real student rejects them.

In [ ]:
rows=experiments(parents,SETTINGS)
for row in rows:print(row['name'],row['architecture'],row['method'],row['head_type'],'encoder view:',row['view'])
if RUN_BENCHMARK:
    measurements=benchmark(rows,DATA_PATH)
    atomic_json(PROJECT/'benchmark.json',measurements)
    print('Measured rough encoder-adaptation compute forecast:',round(sum(x['rough_adaptation_hours'] for x in measurements),2),'hours')
    print('This excludes the other stages and is not a total notebook-runtime prediction.')
if RUN_GPU_SMOKE:smoke_all(PROJECT,rows,DATA_PATH,SETTINGS)

## Train the seven candidates

The optional lexical branch trains on official training cases only. For each candidate, cached Phase-3 features support a warmup baseline. The two-stage route then adapts the encoder on final-window evidence, regenerates all document features from that checkpoint, and trains the complete-document hierarchy. Model selection uses official dev cases. The smaller student receives real two-class Phase-4 teacher distributions after the six larger candidates exist.

Each completed ZIP is saved immediately. If the session stops, rerun using the same files/settings/project/GPU count. Optimizer/scaler/RNG/sampler state is restored. A warmup fallback is explicitly reported if it scores better; examine the actual selected model stages when checking required method diversity. Identical prediction hashes are reported rather than altered.

In [ ]:
lexical=lexical_branch(PROJECT,data_root,DATA_PATH,USE_LEXICAL_FUSION)
catalog=run_all(PROJECT,rows,DATA_PATH,SETTINGS,lexical)
print('Completed base experiments:',len(catalog))
for item in catalog:display(FileLink(item['submission']))

## Score check: full-document joint refinement

Refine the strongest selected candidate using complete-document encoder updates. The existing submission is replaced only after a measured dev improvement; its earlier ZIP is retained. This focuses the expensive joint training on the most promising run. It cannot guarantee a private-test improvement.

An all-label refit is optional and defaults to zero candidates. It uses fixed development-selected epoch counts and adds compute; its dev scores continue to refer to the earlier held-out run. The optional ensemble creates a separate bonus ZIP only when a small, predefined ensemble comparison improves dev. It is not counted as a new adaptation method.

In [ ]:
if JOINT_REFINE_TOP_K:
    catalog=joint_refine_top(PROJECT,rows,DATA_PATH,SETTINGS,lexical,JOINT_REFINE_TOP_K)
if REFIT_TOP_K:
    catalog=refit_top(PROJECT,rows,DATA_PATH,SETTINGS,lexical,REFIT_TOP_K)
if MAKE_BONUS_ENSEMBLE:
    bonus=bonus_ensemble(PROJECT,DATA_PATH,lexical)
    if bonus:display(FileLink(bonus['submission']))
if RUN_SECTION_AUDIT:print(json.dumps(section_audit(PROJECT,DATA_PATH,SETTINGS),indent=2))
print('Portable model manifest:',export_manifest(PROJECT))

## Supplied test labels: final reporting only

The assignment supplies test labels. They are discarded from every model-facing record, so test targets cannot enter training, distillation, dev selection or optional train+dev refit. Only this separate evaluator reads the supplied labels, after all seven predictions and development-based choices are finalized. Test metrics do not change models, thresholds, ensembles or the recommended submission.

Case-level metrics require one unambiguous gold label per original test ID. Conflicting labels under an ID are reported explicitly; no majority vote or official case-level score is invented. A separate diagnostic covers distinct labeled records.

In [ ]:
if EVALUATE_SUPPLIED_TEST_LABELS:
    from cj_test_eval import evaluate_test_labels
    completed=read_json(PROJECT/'catalog.json')
    assert len(completed)==7 and all(Path(item['submission']).is_file() for item in completed)
    reports=[]
    for item in completed:
        report=evaluate_test_labels(data_root,item['submission'],PROJECT/'test_evaluation'/f"{item['name']}.json")
        reports.append({'name':item['name'],**report})
        print(item['name'],'test case metrics:',report['case_metrics'])
        if report['cases_with_conflicting_labels']:
            print('Ambiguous source-case labels:',report['cases_with_conflicting_labels'],
                  'distinct-record diagnostic:',report['distinct_record_diagnostic'])
    atomic_json(PROJECT/'test_evaluation/results.json',reports)
    print('Evaluation only. Existing development-based choices and submissions are unchanged.')

## Exact evaluator checks and measured results

The official script reports Macro-F1 over both classes and binary positive-class precision/recall. Every dev prediction is checked against it. The comparison and plots contain your measured results only. Development scores have been used for checkpoint, threshold and optional fusion/ensemble choices.

In [ ]:
catalog=read_json(PROJECT/'catalog.json')
columns=['name','architecture','method','selected_stage','dev_macro_f1','precision','recall','total_parameters',
         'encoder_stage_trainable','final_stage_trainable','all_label_refit','prediction_sha256']
comparison=[]
for item in catalog:
    result=subprocess.run([sys.executable,str(CODE/'evaluate_phase4_official.py'),'--truth',str(PROJECT/'dev_truth.json'),
                           '--prediction',str(PROJECT/'runs'/item['name']/'dev_prediction.json')],text=True,capture_output=True,check=True)
    print(item['name'],result.stdout.strip().replace('\n',' | '))
    official=float(next(line.split(':')[1] for line in result.stdout.splitlines() if line.startswith('Macro-F1:')))
    assert abs(official-item['dev_metrics']['macro_f1'])<1e-6,'Local metric differs from official evaluation.'
    with zipfile.ZipFile(item['submission']) as archive:
        assert archive.namelist()==['prediction.json']
        prediction=json.loads(archive.read('prediction.json'))
    expected=[data['docs'][i]['id'] for i in data['splits']['test']]
    assert [row['id'] for row in prediction]==expected and len(expected)==len(set(expected))
    assert all(type(row['pred']) is int and row['pred'] in (0,1) for row in prediction)
    row={'name':item['name'],'architecture':item['architecture'],'method':item['method'],'selected_stage':item['selected_stage'],
         'dev_macro_f1':item['dev_metrics']['macro_f1'],'precision':item['dev_metrics']['precision'],'recall':item['dev_metrics']['recall'],
         'total_parameters':item['parameters']['total'],'encoder_stage_trainable':item['encoder_stage_parameters']['trainable'],
         'final_stage_trainable':item['parameters']['trainable'],'all_label_refit':bool(item.get('all_label_refit')),
         'prediction_sha256':item['prediction_sha256']}
    comparison.append(row)
with (PROJECT/'comparison.csv').open('w',newline='') as handle:
    writer=csv.DictWriter(handle,fieldnames=columns);writer.writeheader();writer.writerows(comparison)
print('Distinct prediction sets:',len({item['prediction_sha256'] for item in catalog}),'of 7')
display(FileLink(str(PROJECT/'comparison.csv')))

In [ ]:
import matplotlib.pyplot as plt
plots=PROJECT/'plots';plots.mkdir(exist_ok=True)
names=[item['name'].split('_',1)[0]+' '+item['method'] for item in catalog]
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
axes[0].bar(names,[item['dev_metrics']['macro_f1'] for item in catalog],color='#2b6cb0')
axes[0].set_ylim(0,1);axes[0].set_ylabel('Development Macro-F1 (selected/calibrated)')
axes[0].tick_params(axis='x',rotation=45)
for item in catalog:
    summary=read_json(PROJECT/'runs'/item['name']/'selection/summary.json')
    history=summary['history']
    if history:axes[1].plot([x['epoch'] for x in history],[x['macro_f1'] for x in history],marker='o',label=item['name'][:2])
axes[1].set_xlabel('Encoder-adaptation epoch');axes[1].set_ylabel('Dev Macro-F1 at threshold 0.5')
axes[1].set_ylim(0,1);axes[1].legend(title='Experiment');fig.tight_layout();fig.savefig(plots/'measured_scores.png',dpi=160);plt.show()
best=max(catalog,key=lambda item:item['dev_metrics']['macro_f1'])
matrix=best['dev_metrics']['confusion_matrix']
fig,ax=plt.subplots(figsize=(4.5,4));ax.imshow(matrix,cmap='Blues')
for i in range(2):
    for j in range(2):ax.text(j,i,str(matrix[i][j]),ha='center',va='center')
ax.set(xticks=[0,1],yticks=[0,1],xlabel='Predicted judgment',ylabel='Gold judgment',title=best['name'])
fig.tight_layout();fig.savefig(plots/'best_confusion_matrix.png',dpi=160);plt.show()

## Download predictions and the complete output

Upload the seven individual submission ZIPs, each containing only `prediction.json`. The collection below contains those ZIPs for download and is not itself a platform submission. An optional bonus ensemble is separate. Preserve the complete working archive, including dense parents and encoder-stage checkpoints: the final hierarchical checkpoints are compact and depend on them.

In [ ]:
catalog=read_json(PROJECT/'catalog.json')
if len(catalog)!=7:raise RuntimeError('Complete all seven candidates before exporting the collection.')
collection=WORKING/'phase4_seven_submissions.zip'
with zipfile.ZipFile(collection,'w',zipfile.ZIP_DEFLATED) as archive:
    for item in catalog:archive.write(item['submission'],Path(item['submission']).name)
display(FileLink(str(collection)))
recommendation=PROJECT/'recommended_submission.json'
if recommendation.exists():
    selected=read_json(recommendation)
    print('Development-selected recommended submission:',selected['submission'])
    display(FileLink(selected['submission']))
if MAKE_FULL_WORKING_ARCHIVE:
    complete=bundle_working(WORKING,WORKING/'phase4_all_working.zip')
    display(FileLink(complete))

# Phase 4: court-judgment prediction on Kaggle T4 ×2

This notebook restores your **own Phase-3 outputs**, runs seven adaptation experiments across all four foundation families, and produces binary judgment predictions. The primary metric is **Macro-F1 over labels 0 and 1**. It is a different task and metric from Phase-2 strict entity F1. High private-test performance must be measured; no architecture or paper can guarantee a score for your unseen data and weights.

Requirements were checked against the [current capstone brief](https://docs.google.com/document/d/1olqoT5KPhf5CSJLBNjXGyKyyyQDBr6jcnWqPUh5cNk4/edit), modified October 13, 2026, and the [official Phase-4 evaluator](https://drive.google.com/file/d/1ANCStcPKEJFmGnBQvnf5e7-WFEMy9JE3/view). The Important Instructions require **at least six distinct submissions**, although the Phase-4 section still says four. This notebook runs seven methods and includes mandatory LoRMA and smaller-student distillation. Renamed or identical predictions do not count as distinct submissions. The brief gives **October 21, 2026, 11:59 PM IST** as the Phase-4 deadline, with platform evaluation potentially taking 15–30 minutes or longer. Generating a local ZIP does not upload it to the platform.

The brief restricts LLM-written complete architectures and tuning, and requires an LLM declaration. This deliverable contains AI-written code. Preserve that attribution and follow the course requirements when deciding how to use it. The supplied assignment notebooks are references; their graded blanks have not been filled or modified.

## Input folders

Attach your extracted `phase-3-outputs` Kaggle dataset. Default:

```text
/kaggle/input/datasets/hardik4tiwari/phase-3-outputs/capstone_phase3/
    parents/
        tokenizer.json
        decoder.pt
        encoder_decoder.pt
        local_gqa.pt
        hybrid.pt
    runs/
        01_decoder_full/refit/best.pt
        02_encoder_decoder_full/refit/best.pt
        03_local_gqa_lora/refit/best.pt
        04_hybrid_rslora/refit/best.pt
```

Keep the complete Phase-3 output directory if possible. The four `selection/best.pt` files are accepted when a corresponding refit was not run; every selected path is printed. The small PEFT checkpoints require their dense files in `parents/`. A collection containing only the seven prediction ZIPs is insufficient for training another phase. If your mount is `/kaggle/input/phase-3-outputs/`, edit `PHASE3_ROOT` accordingly. This notebook reads the direct `capstone_phase3` directory; it does not search archives or `all_kaggle_outputs`.

Phase-4 data can be attached separately:

```text
Phase_4_CJPE/
    Train/train.json
    Train/dev.json
    Test/test.json
```

Set `DATA_ROOT` to this directory or its parent. Alternatively, leave it `None`, enable Kaggle Internet, accept the terms of `Exploration-Lab/CS-781-Capstone`, and enable an `HF_TOKEN` Kaggle secret. Dataset revision and file hashes are recorded. Tokens never belong in notebook code. No external model weights, tokenizers, legal corpora or LLM services are downloaded.

The loader accepts record lists, a single record, ID-to-record mappings, Hugging Face column exports, and ID-keyed/pandas columns. Text may be a string or an ordered sentence-string list. Train/dev labels must be scalar integer 0/1; test records may include supplied evaluation labels. Test labels are unconditionally discarded from model-facing records and the token cache. Original test case IDs are preserved exactly, including `.txt` where present. Missing IDs and invalid labels fail rather than being guessed. Repeated train/dev source IDs are resolved into distinct internal records after exact-copy removal. Raw text is tokenized in full. Blank cases retain a representation. Exact training documents duplicated in the development split are removed from training.

### Fix for repeated training case IDs

The official brief says training combines single-instance and multi-instance data. Repeated source case IDs are therefore handled separately from unique prediction IDs. The loader collapses only exact `(original ID, text, label)` copies. Distinct text or labels under one original case ID remain separate training rows with deterministic, collision-checked internal IDs; `source_id` records their original case ID. Differing labels are retained and audited, never silently replaced by the first label or a majority vote. Internal IDs are not appended to the input text. Development records use the same exact-copy/distinct-instance policy, preserving every distinct text/label pair and auditing disagreements. Local development Macro-F1 is measured over these distinct labeled records; it is not a claim about unavailable official case-level truth. For test, exact text copies collapse and distinct texts under one original case ID are concatenated in source order, so every submission still contains one original ID per case. The provided test file may contain labels; model-facing test records always have `gold=None`. Development source cases also found in test are excluded before selection.

All training instances whose original case ID occurs in dev or test are excluded before tokenization. The existing exact-text check also excludes train/dev duplicates under different IDs. The printed `training_id_resolution` audit, including nested `development_id_resolution` and `test_id_resolution`, reports exact copies removed, distinct instances retained, label disagreements, text grouping and held-out case filtering. The lexical and teacher pipelines use the same internal row IDs.

For an existing session failing with a train or dev duplicate case ID (including `2019_890` or `1989_75`), copy the entire contents of `Phase4_Duplicate_ID_Fix_Cell.py` into one cell immediately before the failed data-preparation cell. Run it, then rerun data preparation. It updates the working module and the lexical branch's imported loader reference. Downloaded JSON files, tokenizer and checkpoints are retained; no kernel restart is needed. Do not rerun the old notebook's module-writing cell afterward, because that would overwrite the fix. The updated full notebook already embeds the correction. The traceback establishes the duplicate ID, not whether those particular rows have identical text or labels; the audit resolves that from your real files on Kaggle.

## How the three assignments informed the design

| Supplied notebook | Relevant material | Use here |
|---|---|---|
| `CS781_Assignment2 (1).ipynb` | Bidirectional recurrent encoders, additive attention, native recurrent kernels; cells 58, 163–171, 212–237 | A native two-layer BiGRU over chunk vectors and learned attention pooling; no Python loop over recurrent timesteps |
| `CS781_Assignment_3_Questions.ipynb` | MHA/GQA/MQA, RoPE, mask equivalence, optimized PyTorch attention; cells 17–19, 38–58, 88–104 | Preserve the trained token-backbone architecture and rotary convention; use native SDPA and validate padding instead of converting trained heads to another attention layout |
| `CS781_Assignment_4_Questions_updated.ipynb` | Selective fine-tuning, adapters, efficient LoRMA evaluation, teacher-score caches and forward-KL distillation; cells 117–135, 158–174 | Six adaptation routes and a smaller task-specific student; cached binary teacher logits with the temperature-squared correction |

The assignments are incomplete instructional scaffolds, not trained model checkpoints. They suggest mechanisms, but provide no evidence that a particular mechanism will score highly on CJPE. GPT-2, DistilGPT-2, external GloVe and other pretrained resources mentioned in the assignments are excluded because the capstone requires your own foundation lineage. We do not replace the trained backbone's interleaved RoPE with Assignment 3's split-half convention: those layouts cannot be interchanged silently after training.

## Seven experiments

| Candidate | Phase-3 family | Adaptation of the token encoder | Document head |
|---|---|---|---|
| `01_decoder_full` | Decoder | Full fine-tuning | BiGRU, attention and section pooling |
| `02_encoder_decoder_partial` | Encoder–decoder | Fixed top-two decoder blocks and final normalization | Chunk Transformer and section pooling |
| `03_local_gqa_lorma` | Local GQA | LoRMA+ on attention/feed-forward projections | BiGRU, attention and section pooling |
| `04_decoder_adapter` | Decoder | Residual bottleneck adapters | BiGRU, attention and section pooling |
| `05_hybrid_bitfit` | LSTM–Transformer hybrid | Existing backbone biases only | BiGRU, attention and section pooling |
| `06_encoder_decoder_gradual` | Encoder–decoder | Progressively unfreeze top blocks, then the whole backbone | BiGRU, attention and section pooling |
| `07_decoder_student_kd` | Smaller decoder | Full smaller-student training with hard labels and teacher KL | BiGRU, attention and section pooling |

BitFit uses the hybrid's existing LSTM biases. The Transformer foundations have bias-free projections and RMSNorm, so claiming ordinary BitFit on those would be misleading. LoRMA+ evaluates `y + (alpha/r) B(Ay)` for `y = W x`, without constructing a large square multiplier. The student retains four uniformly spaced blocks from the eight-block decoder; actual counts are logged. A stronger **Phase-4 binary classifier** supplies teacher distributions. Thirteen-class Phase-3 rhetorical-role logits are never treated as judgment targets.

## Architecture and the training-time decision

All token windows are retained in the document representation. Each 512-token window has 64-token overlap by default, and the final window is anchored to the end so final-case evidence is retained in a complete window. Each own-backbone chunk produces masked mean and last-token vectors. A small projection adds continuous document-position features, followed by a two-layer BiGRU or chunk Transformer. The document vector combines learned global attention, maximum pooling, first-quarter mean and last-quarter mean. These final-quarter positions are positional features, not gold rhetorical-role annotations. The classification head produces two logits directly; there is no autoregressive generation or beam search.

The default **two-stage** route follows the task-specific training idea in the ILDC work:

1. Train a full-document head on frozen Phase-3 features. Four parents and the smaller student each get a reusable complete-document feature bank.
2. Adapt the declared token encoder using each case's final window and its document label. This is an explicit context-selection experiment; it avoids seven full-document backward passes at every epoch.
3. Recompute complete-document features from that newly adapted encoder. Train the full-document hierarchy using those fresh features and select on held-out development cases.
4. After seven candidates exist, run full-document joint refinement for the strongest candidate by default. Keep its original submission unless the refined model improves the development selection metric.

Thus final predictions consume the complete case, while expensive encoder updates are concentrated before the final full-document head. This differs from fully joint full-document training for every candidate. It is a research-supported compute choice, not a proof that context selection always preserves accuracy. `two_stage=False` enables full-document joint training for all seven at greater cost. `JOINT_REFINE_TOP_K`, `joint_epochs` and the printed benchmarks make that compute decision visible. No training examples are subsampled and no arbitrary elapsed-time cutoff silently terminates a run. Early stopping uses development Macro-F1.

Whole-document frozen features are stored as FP16 disk-backed arrays; tokens are disk-backed int32. Frozen-head epochs reuse them. After an encoder update, its old feature bank is never used as if it represented the updated encoder: the encoder-checkpoint hash is part of the new cache identity. FP16 autocast and loss scaling, fused AdamW on CUDA, length-grouped head batches, accumulation with DDP `no_sync`, and both GPUs for encoder training/extraction reduce avoidable overhead. Head-only training uses one GPU. Activation checkpointing bounds memory during encoder updates and trades recomputation for memory; it is not inherently a speed improvement. SDPA dispatch is left to PyTorch; no FlashAttention-2 package or unsupported T4 kernel is assumed.

Repeated local masks are cached and padding completeness is checked once per token batch, avoiding a separate GPU-to-host synchronization in every attention layer. Neither changes the attention function.

The word-bigram TF-IDF/logistic branch is fitted **only on official training text and labels**, using full text and a final-text view. Its optional small fusion weight is selected on dev, including a zero-weight option. It is an auxiliary learned branch, not a substitute external foundation or a new required adaptation method. The primary architecture and all seven neural runs retain your own model lineage. Disable `USE_LEXICAL_FUSION` to compare neural-only behavior. Model choice, limited fusion and threshold selection reuse the public dev set; reported scores are development-tuned, not an untouched estimate of private-test performance.

## Research, with implementation boundaries

| Primary source | Relevant finding or mechanism | Concrete use and limitation |
|---|---|---|
| [Malik et al., ILDC for CJPE, ACL 2021](https://aclanthology.org/2021.acl-long.313.pdf), §5.1 and Appendix B.1 | Study final-window adaptation, sliding chunks and hierarchical recurrent document representations | Motivates encoder adaptation followed by complete-document feature extraction and a recurrent head. We use your own backbone and mean/last pooling, not XLNet or its CLS token. Its reported scores are not forecasts for this notebook. |
| [Joshi et al., IL-TUR, 2024](https://arxiv.org/html/2407.05399v2), Appendix B.3 | Compare final-window and hierarchical CJPE representations | Supports measuring whole-document context. External InLegalBERT is prohibited here; its reported 81.31 Macro-F1 cannot be promised with your compact foundation. |
| [Yang et al., HAN, NAACL 2016](https://aclanthology.org/N16-1174.pdf), §2 | Contextual recurrent representations and attention over document units | Motivates chunk-level BiGRU/attention. Token encoding is your own foundation, not HAN's original word-level encoder. Attention weights are not causal explanations. |
| [Bihany, Patel and Modi, LoRMA, ACL Findings 2025](https://aclanthology.org/2025.findings-acl.527.pdf), §3 | Multiplicative low-rank adaptation, rank inflation and reordered evaluation | Implements the additive-identity LoRMA+ form. The permutation variant is not claimed. Low parameter count alone does not imply an equivalent speedup. |
| [Houlsby et al., adapters, ICML 2019](https://arxiv.org/abs/1902.00751), §2 | Small residual bottlenecks with a frozen backbone | Uses one adapter after each backbone block, not the paper's exact two-insertion BERT architecture. Nonlinear adapters stay in their model; they are not falsely merged into a dense weight. |
| [Ben Zaken et al., BitFit](https://arxiv.org/abs/2106.10199) | Tune existing bias terms | Applied to hybrid LSTM biases plus the new task head. Results on BERT do not establish hybrid-CJPE performance. |
| [Howard and Ruder, ULMFiT, ACL 2018](https://aclanthology.org/P18-1031/), §3 | Gradual unfreezing for transfer | Progressively unfreeze the inherited encoder–decoder. Optimizer membership stays fixed and DDP is rebuilt when the trainable set changes. This is not the full ULMFiT recipe. |
| [Hinton et al., distillation, 2015](https://arxiv.org/abs/1503.02531) | Softened teacher targets and temperature-squared KL scaling | Uses cached task-specific binary distributions and hard-label CE. Cache identities include teacher checkpoint, dataset and split. Selection and all-label refits have distinct caches. |
| [Micikevicius et al., mixed precision](https://arxiv.org/abs/1710.03740), [PyTorch AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html) | Reduced-precision matrix operations with scaled gradients | FP16 model operations with FP32 loss calculations, unscaled clipping and skipped-update accounting. T4 does not provide native BF16 support. |
| [Chen et al., activation checkpointing](https://arxiv.org/abs/1604.06174), [PyTorch checkpoint documentation](https://docs.pytorch.org/docs/stable/checkpoint.html) | Recompute activations during backward to reduce storage | Non-reentrant chunk checkpointing. Whether this saves wall time depends on the batch size it permits. |
| [PyTorch SDPA](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention) | Native attention backend selection | Inherited causal, local and cross masks plus explicit padding masks. GQA projections are preserved; portable key/value repetition is not presented as a guaranteed training speedup. |

## Running and preserving the result

Use GPU **T4 ×2**. Run configuration, embedded source, parent restoration and data audit before starting training. The first preflight prints measured memory and a partial compute estimate, then performs a real two-process update for every adaptation method. Benchmark KD uses zero logits only for measurement; actual training rejects a synthetic preflight teacher. The smaller student's real teacher cache is generated after the six larger candidates exist.

The end-to-end time depends on the number and length of cases, early-stopping epochs, I/O and cache reuse. GPU compute estimates exclude full-document feature banks, hierarchy training, validation, teacher inference and file creation. Do not use them as total-runtime promises. Two T4s are separate 16-GB devices; DDP does not combine their memory into one 32-GB device.

Every completed candidate immediately gets its own ZIP, prediction JSON, dev metrics and catalog entry. Recovery checkpoints retain model/deltas, optimizer, scaler, sampler cursor and per-rank RNG state. Keep input files, settings, code and GPU counts fixed when resuming. Use a new PROJECT for a different design. All dense parents, encoder-stage checkpoints and cache manifests must stay with compact hierarchical checkpoints.

`REFIT_TOP_K=0` leaves development labels held out. An optional fixed-epoch all-label refit is provided for selected candidates after seven ZIPs exist; it adds training and has no unbiased development score of its own. Original held-out-model ZIPs are retained as recovery artifacts. `RUN_SECTION_AUDIT` optionally compares all/first/last views of the best selected model. These are post-training sensitivity checks, not independently trained section-only baselines.

Each individual `submissions/<candidate>.zip` contains exactly one root-level `prediction.json`, a JSON list of `{"id": "original_case_id", "pred": 0}` objects. The prediction is an integer 0/1 and every original test ID appears exactly once. `phase4_seven_submissions.zip` is a download collection containing the seven ZIPs; upload the individual ZIPs to the course platform. `phase4_all_working.zip` preserves the complete working directory for download/recovery and later phases.

If the optional, predefined neural ensemble comparison improves dev, it writes a separate `bonus/score_ensemble.zip`; otherwise the recommendation remains an existing candidate. This bonus does not count as another adaptation method. `recommended_submission.json` records the measured choice. `phase4_model_manifest.json` records selected checkpoints and relative dependency paths for restoration from a future Kaggle dataset.

Duplicate prediction hashes are reported without modifying predictions to invent diversity. If a frozen warmup is better on dev, it can be selected as a score safeguard, but the catalog explicitly reports that the submitted model used the warmup. Check diversity among the **actual selected models**, not only the training labels in a filename. Setting `allow_warm_fallback=False` retains each adapted candidate regardless of this measured comparison.

## Verification limits

The real gated Phase-4 data and your actual trained Phase-3 weights were unavailable in this environment. Synthetic own-origin fixtures verify loading and merging of all four source families, seven adaptation gradients, frozen weights, masks/padding, complete token coverage, LoRMA algebra, task-specific KD, cache identities, output ZIP contracts, official metric parity and recovery behavior. These are software checks, not competition scores. Actual T4 timing, two-GPU execution and held-out/private-test quality must be established on Kaggle. The notebook provides the hardware checks and reports rather than inventing those results.


### Supplied test labels: evaluation only

Train fits weights; dev selects checkpoints, epochs, thresholds, fusion and ensemble choices. Optional refit uses train+dev only. Test texts are encoded for inference, but every test target in the model cache is `None` (batched as ignore index `-100`). Test IDs are disjoint from training/refit source cases. Supplied test labels are read by a separate evaluator only after all seven submissions and optional development-selected refinements are finalized. Test results do not change the catalog, recommended submission, model, threshold or ensemble.

The evaluator validates prediction coverage. When all records under an original test ID agree on its label, it reports case-level Macro-F1, positive-class precision/recall and confusion matrix, with one case per prediction ID. It also reports a diagnostic over distinct labeled records. If a test ID has conflicting labels, case-level truth is ambiguous: no label vote is made, no official case-level score is claimed, and the diagnostic and conflicting IDs are reported explicitly.

For the `test labels must not be supplied` error, run `Phase4_Test_Label_Fix_Cell.py` after the earlier duplicate-ID repair and before data preparation. It changes the test parser, writes the repaired module and reloads active functions; no download or restart is needed. An updated full duplicate-ID repair and full notebook also include this correction.

### Hybrid BitFit activation-checkpoint repair

A Kaggle T4 ×2 log passed the first four distributed updates, then failed in `05_hybrid_bitfit` with `CheckpointError` reporting differing saved/recomputed tensor metadata. The failing path wrapped the complete hybrid encoder, including its fused cuDNN LSTM, in a non-reentrant activation checkpoint. This points to the recurrent mixed-precision recomputation path; its exact CUDA kernel cause has not been reproduced in the CPU development environment. PyTorch documents that checkpointing reruns forward operations during backward, and its tracker includes related [autocast/checkpoint metadata failures](https://github.com/pytorch/pytorch/issues/141896).

The repaired hybrid path computes the same embedding → residual LSTM → Transformer blocks → final norm operation. The LSTM runs once per chunk group outside checkpoint recomputation; only Transformer blocks are checkpointed. Each replay explicitly restores that chunk's padding mask, including batches with several chunk groups and different sequence lengths. The task head, existing BitFit biases, parameter counts, loss, epochs, dropout and mixed precision remain the same. The fused recurrent backend stays enabled. Other architecture paths are unchanged; test labels remain evaluation-only.

For a running notebook, copy all of `Phase4_Hybrid_Checkpoint_Fix_Cell.py` into a new Code cell immediately before the failed preflight cell and run it. It updates `cj_model.py` on disk for worker processes and refreshes the notebook's imported model builder. Do not rerun old module-writing cells afterward. Rerun `smoke_all(PROJECT, rows, DATA_PATH, SETTINGS)` without repeating the benchmark or data preparation, and continue training only after all seven actual distributed updates pass. Keep the same project/configuration so downloaded data and feature caches are retained. CPU parity checks do not certify CUDA mixed-precision DDP behavior; the Kaggle preflight is the required GPU check.

Validation for this repair: all seven methods matched uncheckpointed CPU forward outputs and trainable gradients within numerical tolerance, preserved frozen parameters and observable RNG state, and completed real single-process CPU optimizer updates in worker subprocesses. The hybrid LSTM was called once per chunk group, and the live repair was repeatable. This environment has no CUDA and disallows the local sockets needed by CPU/Gloo distributed tests, so neither the T4 mixed-precision test nor the two-process check is claimed as locally passed.